# Dose-Stratified Transcriptomic Reversal

In [ ]:
# -*- coding: utf-8 -*-
# @title Study 1 (v2 Enhanced): Dose- & Tissue-Stratified Montelukast L1000 Reversal vs. AD Skin & Clinical Benchmarks (Single-Cell Colab Pipeline)
"""
Single-cell, self-contained Google Colab pipeline for Study 1 (v2 Enhanced).

Enhancements over v1:
  - Plan 1 (Low-Dose Stratum): Unions SigCom LINCS pert_name + pert_id (BRD IDs & salt forms)
    with pagination, plus 2-tier on-target CysLT1R proxies (Zafirlukast, Pranlukast, and
    CYSLTR1 CRISPR/shRNA knockdown in L1000) if Montelukast <= 1 uM remains sparse.
  - Plan 2 (GSE130588 Fix): Structured key-value parser on !Sample_characteristics_ch1
    (tissue, time, treatment, SCORAD) without scanning free-text description, plus strict
    arm-size QC checks and sample-level verification logging.
  - Plan 3 (Zero Pathway Gaps): Adds QuickGO REST API (goUsage=descendants) for empty
    Enrichr GO:BP terms and pairs GO:0031424 / GO:0042092 with Reactome canonical pathways
    R-HSA-6805567 (Keratinization) and R-HSA-6785807 (IL-4 and IL-13 signaling).
  - Plan 4 (Cell-Line Confounding & Statistical Rigor): Tissue-relevance stratification
    (Cutaneous/Epithelial/Immune vs. Hormone-Driven Tumors vs. Pan-Cell), vectorized
    2,000-permutation GSEA WTCS, and Benjamini-Hochberg FDR across all tests.
"""

import gzip
import io
import json
import os
import re
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from scipy.spatial.distance import cosine
from scipy.stats import spearmanr, ttest_ind
from urllib3.util.retry import Retry

# =============================================================================
# 1. CONFIGURATION & STANDARDIZED PATHWAY / TISSUE DEFINITIONS
# =============================================================================
os.makedirs("data/pathways", exist_ok=True)
os.makedirs("results", exist_ok=True)

MOUNT_DRIVE = False               # @param {type:"boolean"}
DRIVE_EXPORT_DIR = ""             # @param {type:"string"}  # e.g. "/content/drive/MyDrive/AD_Study1_v2"
LOW_DOSE_MAX_UM = 1.0             # @param {type:"number"}
HIGH_DOSE_MIN_UM = 10.0           # @param {type:"number"}
MAX_SIGS_PER_STRATUM = 100        # @param {type:"integer"}
TOP_N_DEG_GSEA = 150              # @param {type:"integer"}
GSEA_PERMUTATIONS = 2000          # @param {type:"integer"}  # Plan 4: 2,000 vectorized permutations
RANDOM_SEED = 42                  # @param {type:"integer"}
ENABLE_ILIKE_FALLBACK = False     # @param {type:"boolean"}
SCORE_INDIVIDUAL_PATHWAYS = True  # @param {type:"boolean"}

IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if MOUNT_DRIVE and IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)

# Plan 3: Standardized GO:BP + KEGG + Reactome canonical pathways
AD_PATHWAY_SPECS = {
    "Type2_Inflammation": {
        "GOBP": [
            ("GO:0042092", "type 2 immune response"),
            ("GO:0002828", "regulation of type 2 immune response"),
        ],
        "KEGG": [
            ("hsa04658", "Th1 and Th2 cell differentiation"),
            ("hsa04664", "Fc epsilon RI signaling pathway"),
        ],
        "REACTOME": [
            ("R-HSA-6785807", "Interleukin-4 and Interleukin-13 signaling"),
        ],
    },
    "Epidermal_Barrier": {
        "GOBP": [
            ("GO:0061436", "establishment of skin barrier"),
            ("GO:0031424", "keratinization"),
            ("GO:0030216", "keratinocyte differentiation"),
            ("GO:0008544", "epidermis development"),
        ],
        "KEGG": [
            ("hsa04530", "Tight junction"),
            ("hsa00062", "Fatty acid elongation"),
            ("hsa00600", "Sphingolipid metabolism"),
        ],
        "REACTOME": [
            ("R-HSA-6805567", "Keratinization"),
        ],
    },
    "AMP_Innate": {
        "GOBP": [
            ("GO:0019730", "antimicrobial humoral response"),
            ("GO:0061844", "antimicrobial humoral immune response mediated by antimicrobial peptide"),
            ("GO:0019731", "antibacterial humoral response"),
        ],
        "KEGG": [
            ("hsa04657", "IL-17 signaling pathway"),
        ],
        "REACTOME": [],
    },
    "cAMP_PDE4_5LOX_Panel": {
        "GOBP": [],
        "KEGG": [
            ("hsa04024", "cAMP signaling pathway"),
            ("hsa00590", "Arachidonic acid metabolism"),
        ],
        "REACTOME": [],
    },
}

# Plan 4: L1000 Tissue-Relevance Cell-Line Strata
RELEVANT_TISSUE_CELLS = {
    "A375", "HA1E", "A549", "HME1", "MCF10A", "HEK293", "HEKTE",
    "THP1", "JURKAT", "U937", "HL60", "NOMO1", "WSUDLCL2", "U2OS",
}
HORMONE_TUMOR_CELLS = {
    "MCF7", "VCAP", "PC3", "HT29", "HCC515", "HEPG2", "LNCAP", "SKBR3", "BT20", "MDAMB231",
}

GENE_ALIAS_NORMALIZATION = {
    "LORICRIN": "LOR",
    "DEFB4": "DEFB4A",
    "DEFB2": "DEFB4A",
    "HBD2": "DEFB4A",
    "LL37": "CAMP",
    "CAP18": "CAMP",
    "IL4RA": "IL4R",
}

SPOTLIGHT_GENES = [
    "IL4", "IL13", "IL4R", "CCL17", "CCL18", "CCL22", "CCL26", "POSTN", "MMP12", "SERPINB4", "KRT16",
    "FLG", "LOR", "IVL", "ELOVL3", "ELOVL6", "CLDN8", "CLDN1", "LCE2B", "FADS1", "CERS3",
    "CAMP", "DEFB4A", "DEFB103A", "S100A7", "S100A8", "S100A9", "S100A12", "PI3", "LCN2",
    "PDE4A", "PDE4B", "PDE4D", "NR4A1", "NR4A2", "NR4A3", "DUSP1", "CREM", "ALOX5", "LTC4S", "CYSLTR1",
]

def normalize_symbol(sym):
    s = str(sym).strip().upper()
    return GENE_ALIAS_NORMALIZATION.get(s, s)

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "Study1-Montelukast-AD-L1000-Benchmark/2.1"})
_retry = Retry(
    total=5,
    backoff_factor=3,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET", "POST"],
)
_adapter = HTTPAdapter(max_retries=_retry)
SESSION.mount("https://", _adapter)
SESSION.mount("http://", _adapter)


# =============================================================================
# 2. STANDARDIZED GO:BP + KEGG + REACTOME PATHWAY RESOLVER (PLAN 3)
# =============================================================================
ENRICHR_GMT_URL = "https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName={lib}"
KEGG_REST_GET = "https://rest.kegg.jp/get/{kegg_id}"
QUICKGO_DOWNLOAD_URL = "https://www.ebi.ac.uk/QuickGO/services/annotation/downloadSearch"
REACTOME_REF_ENTITIES_URL = "https://reactome.org/ContentService/data/participants/{reactome_id}/referenceEntities"

def parse_kegg_flatfile_genes(text):
    genes = []
    in_gene = False
    for line in text.splitlines():
        if line.startswith("GENE"):
            in_gene = True
            payload = line[12:].strip()
        elif in_gene:
            if line and not line[0].isspace():
                break
            payload = line.strip()
        else:
            continue
        if not payload:
            continue
        parts = payload.split(None, 1)
        if len(parts) < 2:
            continue
        sym_part = parts[1].split(";")[0].split(",")[0].strip()
        if sym_part and re.match(r"^[A-Za-z0-9_.-]+$", sym_part):
            genes.append(normalize_symbol(sym_part))
    return sorted(set(genes))

def parse_enrichr_gmt_by_go_id(gmt_text):
    go_map = {}
    for line in gmt_text.splitlines():
        if not line.strip():
            continue
        parts = line.rstrip("\r\n").split("\t")
        if len(parts) < 3:
            continue
        m = re.search(r"\((GO:\d{7})\)", parts[0].strip())
        if not m:
            continue
        genes = sorted({normalize_symbol(g.split(",")[0]) for g in parts[2:] if g.strip()})
        if genes:
            go_map[m.group(1)] = genes
    return go_map

def parse_enrichr_gmt_by_name(gmt_text):
    name_map = {}
    for line in gmt_text.splitlines():
        if not line.strip():
            continue
        parts = line.rstrip("\r\n").split("\t")
        if len(parts) < 3:
            continue
        term = parts[0].strip().lower()
        genes = sorted({normalize_symbol(g.split(",")[0]) for g in parts[2:] if g.strip()})
        if genes:
            name_map[term] = genes
    return name_map

def parse_quickgo_tsv_genes(tsv_text):
    """Parse QuickGO annotation/downloadSearch TSV response and extract unique HGNC SYMBOL values."""
    lines = [ln for ln in tsv_text.splitlines() if ln.strip() and not ln.startswith("#")]
    if len(lines) < 2:
        return []
    header = [h.strip().upper() for h in lines[0].split("\t")]
    sym_idx = next((i for i, h in enumerate(header) if h in {"SYMBOL", "GENE_PRODUCT_SYMBOL", "DB_OBJECT_SYMBOL"}), None)
    if sym_idx is None:
        return []
    genes = set()
    for ln in lines[1:]:
        parts = ln.split("\t")
        if len(parts) > sym_idx:
            s = parts[sym_idx].strip()
            if s and s != "-" and re.match(r"^[A-Za-z0-9_.-]+$", s):
                genes.add(normalize_symbol(s))
    return sorted(genes)

def parse_reactome_reference_entities(json_data):
    """Extract unique human HGNC gene symbols from Reactome Content Service referenceEntities JSON."""
    if not isinstance(json_data, list):
        return []
    genes = set()
    for ent in json_data:
        if not isinstance(ent, dict):
            continue
        gn = ent.get("geneName")
        if isinstance(gn, list) and gn:
            first = str(gn[0]).strip()
            if first and re.match(r"^[A-Za-z0-9_.-]+$", first):
                genes.add(normalize_symbol(first))
        elif isinstance(gn, str) and gn.strip():
            genes.add(normalize_symbol(gn.strip()))
    return sorted(genes)

def load_enrichr_gobp_library(cache_dir="data/pathways"):
    for lib in ["GO_Biological_Process_2025", "GO_Biological_Process_2023", "GO_Biological_Process_2021"]:
        cache_file = Path(cache_dir) / f"{lib}.gmt"
        if cache_file.exists() and cache_file.stat().st_size > 1000:
            return parse_enrichr_gmt_by_go_id(cache_file.read_text(encoding="utf-8", errors="replace")), lib
        try:
            r = SESSION.get(ENRICHR_GMT_URL.format(lib=lib), timeout=90)
            if r.ok and len(r.text) > 10000 and "(GO:" in r.text:
                cache_file.write_text(r.text, encoding="utf-8")
                return parse_enrichr_gmt_by_go_id(r.text), lib
        except Exception as e:
            print(f"    [WARN] Enrichr {lib} fetch failed: {e}")
    return {}, "unavailable"

def load_enrichr_kegg_library(cache_dir="data/pathways"):
    for lib in ["KEGG_2021_Human", "KEGG_2019_Human"]:
        cache_file = Path(cache_dir) / f"{lib}.gmt"
        if cache_file.exists() and cache_file.stat().st_size > 1000:
            return parse_enrichr_gmt_by_name(cache_file.read_text(encoding="utf-8", errors="replace")), lib
        try:
            r = SESSION.get(ENRICHR_GMT_URL.format(lib=lib), timeout=90)
            if r.ok and len(r.text) > 5000:
                cache_file.write_text(r.text, encoding="utf-8")
                return parse_enrichr_gmt_by_name(r.text), lib
        except Exception as e:
            print(f"    [WARN] Enrichr {lib} fetch failed: {e}")
    return {}, "unavailable"

def fetch_gobp_genes_quickgo(go_id, cache_dir="data/pathways"):
    """Plan 3: Official QuickGO REST API with goUsage=descendants (includes child GO terms)."""
    cache_file = Path(cache_dir) / f"quickgo_{go_id.replace(':', '_')}.tsv"
    if cache_file.exists() and cache_file.stat().st_size > 100:
        genes = parse_quickgo_tsv_genes(cache_file.read_text(encoding="utf-8", errors="replace"))
        if genes:
            return genes, "QuickGO_descendants_cached"
    try:
        r = SESSION.get(
            QUICKGO_DOWNLOAD_URL,
            params={
                "goId": go_id,
                "taxonId": "9606",
                "goUsage": "descendants",
                "geneProductType": "protein",
                "selectedFields": "symbol",
            },
            headers={"Accept": "text/tsv"},
            timeout=90,
        )
        if r.ok and "SYMBOL" in r.text[:500].upper():
            cache_file.write_text(r.text, encoding="utf-8")
            genes = parse_quickgo_tsv_genes(r.text)
            if genes:
                return genes, "QuickGO_descendants"
    except Exception as e:
        print(f"    [WARN] QuickGO fallback failed for {go_id}: {e}")
    return [], "FAILED"

def fetch_gobp_genes_mygene(go_id):
    try:
        r = SESSION.get(
            "https://mygene.info/v3/query",
            params={"q": f'go.BP.id:"{go_id}"', "species": "human", "fields": "symbol", "size": 1000},
            timeout=45,
        )
        if r.ok:
            hits = r.json().get("hits", [])
            return sorted({normalize_symbol(h["symbol"]) for h in hits if isinstance(h, dict) and h.get("symbol")})
    except Exception as e:
        print(f"    [WARN] MyGene GO fallback failed for {go_id}: {e}")
    return []

def fetch_kegg_pathway_genes(kegg_id, kegg_name, enrichr_kegg_map=None, cache_dir="data/pathways"):
    cache_file = Path(cache_dir) / f"{kegg_id}.txt"
    if cache_file.exists() and cache_file.stat().st_size > 200:
        genes = parse_kegg_flatfile_genes(cache_file.read_text(encoding="utf-8", errors="replace"))
        if genes:
            return genes, "KEGG_REST_cached"
    try:
        r = SESSION.get(KEGG_REST_GET.format(kegg_id=kegg_id), timeout=45)
        if r.ok and "GENE" in r.text:
            cache_file.write_text(r.text, encoding="utf-8")
            genes = parse_kegg_flatfile_genes(r.text)
            if genes:
                return genes, "KEGG_REST"
    except Exception as e:
        print(f"    [WARN] KEGG REST fetch failed for {kegg_id}: {e}")

    if enrichr_kegg_map:
        g = enrichr_kegg_map.get(kegg_name.lower(), [])
        if g:
            return g, "Enrichr_KEGG_fallback"
    return [], "FAILED"

def fetch_reactome_pathway_genes(reactome_id, reactome_name, cache_dir="data/pathways"):
    """Plan 3: Official Reactome Content Service REST API for canonical human pathways."""
    cache_file = Path(cache_dir) / f"{reactome_id}.json"
    if cache_file.exists() and cache_file.stat().st_size > 50:
        try:
            genes = parse_reactome_reference_entities(json.loads(cache_file.read_text(encoding="utf-8")))
            if genes:
                return genes, "Reactome_REST_cached"
        except Exception:
            pass
    try:
        r = SESSION.get(
            REACTOME_REF_ENTITIES_URL.format(reactome_id=reactome_id),
            headers={"Accept": "application/json"},
            timeout=60,
        )
        if r.ok:
            data = r.json()
            cache_file.write_text(json.dumps(data), encoding="utf-8")
            genes = parse_reactome_reference_entities(data)
            if genes:
                return genes, "Reactome_REST"
    except Exception as e:
        print(f"    [WARN] Reactome REST fetch failed for {reactome_id}: {e}")

    # Fallback: Enrichr Reactome_2022 GMT
    try:
        lib = "Reactome_2022"
        gmt_cache = Path(cache_dir) / f"{lib}.gmt"
        if not gmt_cache.exists():
            r = SESSION.get(ENRICHR_GMT_URL.format(lib=lib), timeout=90)
            if r.ok and len(r.text) > 5000:
                gmt_cache.write_text(r.text, encoding="utf-8")
        if gmt_cache.exists():
            for line in gmt_cache.read_text(encoding="utf-8", errors="replace").splitlines():
                if reactome_id in line:
                    parts = line.rstrip("\r\n").split("\t")
                    genes = sorted({normalize_symbol(g.split(",")[0]) for g in parts[2:] if g.strip()})
                    if genes:
                        return genes, "Enrichr_Reactome_2022"
    except Exception:
        pass
    return [], "FAILED"

def build_standardized_ad_submodules(pathway_specs=AD_PATHWAY_SPECS):
    print("  - Loading standardized GO:BP + KEGG + Reactome pathway libraries...")
    gobp_map, gobp_lib = load_enrichr_gobp_library()
    kegg_fallback_map = None

    ad_submodules_norm = {}
    individual_pathways = {}
    inv_rows = []
    gene_prov = {}

    for mod_name, db_dict in pathway_specs.items():
        mod_genes = set()
        # 1) GO:BP (Enrichr -> QuickGO descendants -> MyGene)
        for go_id, go_name in db_dict.get("GOBP", []):
            genes = gobp_map.get(go_id, [])
            src = f"Enrichr:{gobp_lib}" if genes else ""
            if not genes:
                genes, src = fetch_gobp_genes_quickgo(go_id)
            if not genes:
                genes = fetch_gobp_genes_mygene(go_id)
                src = "MyGene.info" if genes else "FAILED"
            inv_rows.append({
                "module": mod_name, "database": "GO:BP", "pathway_id": go_id,
                "pathway_name": go_name, "n_genes": len(genes), "source": src,
            })
            if genes:
                mod_genes.update(genes)
                individual_pathways[f"{go_id}_{go_name[:28].replace(' ', '_')}"] = genes
                for g in genes:
                    gene_prov.setdefault((mod_name, g), set()).add(go_id)

        # 2) KEGG (KEGG REST -> Enrichr KEGG)
        for kegg_id, kegg_name in db_dict.get("KEGG", []):
            genes, src = fetch_kegg_pathway_genes(kegg_id, kegg_name, enrichr_kegg_map=kegg_fallback_map)
            if not genes and kegg_fallback_map is None:
                kegg_fallback_map, _ = load_enrichr_kegg_library()
                genes, src = fetch_kegg_pathway_genes(kegg_id, kegg_name, enrichr_kegg_map=kegg_fallback_map)
            inv_rows.append({
                "module": mod_name, "database": "KEGG", "pathway_id": kegg_id,
                "pathway_name": kegg_name, "n_genes": len(genes), "source": src,
            })
            if genes:
                mod_genes.update(genes)
                individual_pathways[f"{kegg_id}_{kegg_name[:28].replace(' ', '_')}"] = genes
                for g in genes:
                    gene_prov.setdefault((mod_name, g), set()).add(kegg_id)
            time.sleep(0.15)

        # 3) Reactome (Reactome Content Service REST -> Enrichr Reactome_2022)
        for r_id, r_name in db_dict.get("REACTOME", []):
            genes, src = fetch_reactome_pathway_genes(r_id, r_name)
            inv_rows.append({
                "module": mod_name, "database": "REACTOME", "pathway_id": r_id,
                "pathway_name": r_name, "n_genes": len(genes), "source": src,
            })
            if genes:
                mod_genes.update(genes)
                individual_pathways[f"{r_id}_{r_name[:28].replace(' ', '_')}"] = genes
                for g in genes:
                    gene_prov.setdefault((mod_name, g), set()).add(r_id)
            time.sleep(0.15)

        ad_submodules_norm[mod_name] = sorted(mod_genes)
        print(f"    [OK] {mod_name:22s}: {len(mod_genes):4d} unique standardized genes")

    inv_df = pd.DataFrame(inv_rows)
    prov_df = pd.DataFrame([
        {"module": mod, "symbol": g, "pathway_ids": ";".join(sorted(pids))}
        for (mod, g), pids in sorted(gene_prov.items())
    ])
    return ad_submodules_norm, individual_pathways, inv_df, prov_df


# =============================================================================
# 3. SIGCOM LINCS L1000 UNION RETRIEVAL + DOSE & TISSUE STRATIFICATION (PLANS 1 & 4)
# =============================================================================
SIGCOM_META = "https://maayanlab.cloud/sigcom-lincs/metadata-api"
SIGCOM_DATA = "https://maayanlab.cloud/sigcom-lincs/data-api/api/v1"
GENETIC_PERT_TYPES = {"CRISPR", "SHRNA", "OVEREXPRESSION", "TRT_SH", "TRT_OE", "TRT_XPR", "TRT_LIG"}
KD_PERT_TYPES = {"CRISPR", "SHRNA", "TRT_SH", "TRT_XPR", "CRISPR KNOCKDOWN", "SHRNA KNOCKDOWN"}

PERTURBAGENS = {
    "Montelukast": {
        "names": ["montelukast", "montelukast sodium", "montelukast-sodium", "MK-0476", "MK-476", "singulair"],
        "pert_ids": [
            "BRD-K61868913", "BRD-A61868913",
            "BRD-K61868913-001-01-7", "BRD-K61868913-236-01-9",
            "BRD-A61868913-001-01-7", "BRD-A61868913-236-01-9",
        ],
        "smiles": [
            "CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cccc(/C=C/c2ccc3ccc(Cl)cc3n2)c1",
        ],
        "stratify_dose": True,
        "allow_genetic": False,
        "role": "Test_Compound",
    },
    # Plan 1: 2-Tier On-Target CysLT1R Fallback Proxies
    "Zafirlukast": {
        "names": ["zafirlukast", "ICI-204219", "accolate"],
        "pert_ids": ["BRD-K01658272", "BRD-A01658272"],
        "stratify_dose": True,
        "allow_genetic": False,
        "role": "Proxy_CysLT1R_Antagonist",
    },
    "Pranlukast": {
        "names": ["pranlukast", "ONO-1078", "onon"],
        "pert_ids": ["BRD-K77007512", "BRD-A77007512"],
        "stratify_dose": True,
        "allow_genetic": False,
        "role": "Proxy_CysLT1R_Antagonist",
    },
    "CYSLTR1_KD": {
        "names": ["CYSLTR1", "cysltr1"],
        "pert_ids": [],
        "stratify_dose": False,
        "allow_genetic": True,
        "genetic_only_kd": True,
        "role": "Proxy_CysLT1R_GeneticKD",
    },
    # Mechanistic Controls (5-LOX & PDE4)
    "Zileuton": {
        "names": ["zileuton", "A-64077", "zyflo"],
        "pert_ids": ["BRD-K04548931", "BRD-A04548931"],
        "stratify_dose": False,
        "allow_genetic": False,
        "role": "Control_5LOXi",
    },
    "Roflumilast": {
        "names": ["roflumilast", "APTA-2217", "daliresp"],
        "pert_ids": ["BRD-K74142843"],
        "stratify_dose": False,
        "allow_genetic": False,
        "role": "Control_PDE4i",
    },
    "Crisaborole": {
        "names": ["crisaborole", "AN2728", "AN-2728", "eucrisa"],
        "pert_ids": ["BRD-K00003448"],
        "stratify_dose": False,
        "allow_genetic": False,
        "role": "Control_PDE4i_Topical",
    },
    "Rolipram": {
        "names": ["rolipram", "ZK-62711"],
        "pert_ids": ["BRD-K71035033", "BRD-A71035033"],
        "stratify_dose": False,
        "allow_genetic": False,
        "role": "Control_PDE4i_Ref",
    },
    "Apremilast": {
        "names": ["apremilast", "CC-10004", "otezla"],
        "pert_ids": ["BRD-K20168484"],
        "stratify_dose": False,
        "allow_genetic": False,
        "role": "Control_PDE4i_Clinical",
    },
}

def parse_dose_um(meta_dict):
    if not isinstance(meta_dict, dict):
        return np.nan
    raw_dose = (
        meta_dict.get("pert_dose")
        or meta_dict.get("pert_idose")
        or meta_dict.get("concentration")
        or meta_dict.get("dose")
    )
    raw_unit = str(meta_dict.get("pert_dose_unit") or "").strip().lower()
    if raw_dose is None:
        return np.nan
    s = str(raw_dose).strip()
    if not s or s in {"-666", "nan", "none", "na", "null", "unknown"}:
        return np.nan
    m = re.search(r"([0-9]+(?:\.[0-9]+)?(?:[eE][+-]?[0-9]+)?)\s*([a-zA-Zµμ]+)?", s)
    if not m:
        return np.nan
    val = float(m.group(1))
    if val < 0:
        return np.nan
    unit = (m.group(2) or raw_unit or "um").strip().lower().replace("µ", "u").replace("μ", "u")
    if unit in {"um", "umol", "umol/l", "micromolar"}:
        return val
    if unit in {"nm", "nmol", "nmol/l", "nanomolar"}:
        return val / 1000.0
    if unit in {"mm", "mmol", "mmol/l", "millimolar"}:
        return val * 1000.0
    if unit in {"pm", "pmol", "picomolar"}:
        return val / 1e6
    if unit in {"m", "mol", "mol/l"}:
        return val * 1e6
    return val

def classify_dose_bin(dose_um, low_max=LOW_DOSE_MAX_UM, high_min=HIGH_DOSE_MIN_UM):
    if pd.isna(dose_um):
        return "Unknown_Dose"
    if dose_um <= low_max + 1e-9:
        return "Low_le_1uM"
    if dose_um >= high_min - 1e-9:
        return "High_ge_10uM"
    return "Mid_1_to_10uM"

def classify_cell_tissue(cell_line):
    c = str(cell_line or "").strip().upper()
    if c in RELEVANT_TISSUE_CELLS:
        return "RelevantTissue"
    if c in HORMONE_TUMOR_CELLS:
        return "HormoneTumor"
    return "OtherCell"

def _sort_sigs_deterministic(sigs):
    return sorted(
        sigs,
        key=lambda s: (
            str(s.get("id", "")),
            str((s.get("meta") or {}).get("persistent_id", "")),
        ),
    )

def _sigcom_find_paginated(where_clause, page_size=500, max_pages=4):
    """Run paginated POST /signatures/find against SigCom LINCS Metadata API."""
    all_rows = []
    for page in range(max_pages):
        try:
            r = SESSION.post(
                f"{SIGCOM_META}/signatures/find",
                json={"filter": {"where": where_clause, "limit": page_size, "skip": page * page_size}},
                timeout=90,
            )
            r.raise_for_status()
            batch = r.json()
            if not isinstance(batch, list) or not batch:
                break
            all_rows.extend(batch)
            if len(batch) < page_size:
                break
        except requests.exceptions.RequestException as e:
            print(f"    [WARN] SigCom query error ({list(where_clause.keys())}, page {page}): {e}")
            break
    return all_rows

def query_sigcom_signatures(spec):
    """
    Plan 1: ALWAYS union meta.pert_name + meta.pert_id + meta.canonical_smiles queries
    (never skipping pert_id when pert_name is non-empty) and deduplicate by signature ID.
    """
    pert_names = spec.get("names", [])
    pert_ids = spec.get("pert_ids", [])
    smiles = spec.get("smiles", [])
    allow_genetic = bool(spec.get("allow_genetic", False))
    genetic_only_kd = bool(spec.get("genetic_only_kd", False))

    found = []
    if pert_names:
        variants = []
        for name in pert_names:
            variants.extend([name, name.lower(), name.upper(), name.title()])
        variants = list(dict.fromkeys(variants))
        found.extend(_sigcom_find_paginated({"meta.pert_name": {"inq": variants}}))

    if pert_ids:
        found.extend(_sigcom_find_paginated({"meta.pert_id": {"inq": pert_ids}}))

    if smiles:
        found.extend(_sigcom_find_paginated({"meta.canonical_smiles": {"inq": smiles}}))

    if not found and ENABLE_ILIKE_FALLBACK and pert_names:
        found.extend(_sigcom_find_paginated({"meta.pert_name": {"ilike": f"%{pert_names[0]}%"}}, max_pages=1))

    dedup = {}
    for s in found:
        sid = str(s.get("id", ""))
        if not sid:
            continue
        ptype = str((s.get("meta") or {}).get("pert_type", "")).upper()
        if genetic_only_kd:
            if any(k in ptype for k in ["CRISPR", "SHRNA", "TRT_SH", "TRT_XPR"]):
                dedup[sid] = s
        elif not allow_genetic:
            if ptype not in GENETIC_PERT_TYPES and not any(k in ptype for k in ["CRISPR", "SHRNA", "OVEREXPRESSION"]):
                dedup[sid] = s
        else:
            dedup[sid] = s
    return _sort_sigs_deterministic(list(dedup.values()))

def parse_persistent_tsv(url):
    resp = SESSION.get(url, timeout=45)
    resp.raise_for_status()
    lines = resp.text.splitlines()
    if len(lines) < 10:
        return pd.Series(dtype=float)
    first_parts = lines[0].split("\t")
    has_header = False
    if len(first_parts) >= 2:
        try:
            float(first_parts[1])
        except ValueError:
            has_header = True
    df = pd.read_csv(
        io.StringIO(resp.text),
        sep="\t",
        header=0 if has_header else None,
        usecols=[0, 1],
        names=["symbol", "val"] if not has_header else None,
    )
    df.columns = ["symbol", "val"]
    df["symbol"] = df["symbol"].astype(str).map(normalize_symbol)
    df["val"] = pd.to_numeric(df["val"], errors="coerce")
    return df.dropna(subset=["symbol", "val"]).groupby("symbol")["val"].mean()

def zscore_series(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    if s.empty:
        return s
    std = s.std(ddof=0)
    if pd.isna(std) or std == 0:
        return s
    return (s - s.mean()) / std

def collapse_signatures_to_consensus(sig_records, max_sigs=MAX_SIGS_PER_STRATUM):
    chosen = _sort_sigs_deterministic(sig_records)[:max_sigs]
    if not chosen:
        return pd.Series(dtype=float), [], 0

    vectors = []
    used_meta = []
    for s in chosen:
        meta = s.get("meta") or {}
        pid_url = meta.get("persistent_id") or s.get("persistent_id")
        if pid_url and str(pid_url).startswith("http"):
            try:
                v = parse_persistent_tsv(pid_url)
                if not v.empty:
                    vectors.append(v)
                    used_meta.append(s)
            except Exception:
                continue

    if vectors:
        mat = pd.concat(vectors, axis=1)
        return zscore_series(mat.median(axis=1)), used_meta, len(vectors)

    try:
        sig_ids = [s["id"] for s in chosen if "id" in s]
        resp = SESSION.post(f"{SIGCOM_DATA}/fetch", json={"signatures": sig_ids}, timeout=90)
        if resp.ok:
            data_json = resp.json()
            if isinstance(data_json, dict) and "data" in data_json:
                mat = pd.DataFrame(data_json["data"])
                mat.index = mat.index.astype(str).map(normalize_symbol)
                mat = mat.groupby(level=0).mean()
                return zscore_series(mat.median(axis=1)), chosen[:mat.shape[1]], mat.shape[1]
    except Exception:
        pass

    return pd.Series(dtype=float), [], 0


# =============================================================================
# 4. NCBI GEO AD COHORTS & STRUCTURED METADATA PARSER (PLAN 2)
# =============================================================================
GEO_URLS = {
    "GSE130588": {
        "type": "microarray_gpl570",
        "matrix": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE130nnn/GSE130588/matrix/GSE130588_series_matrix.txt.gz",
    },
    "GSE133477": {
        "type": "microarray_gpl570",
        "matrix": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE133nnn/GSE133477/matrix/GSE133477_series_matrix.txt.gz",
    },
    "GSE121212": {
        "type": "rnaseq",
        "matrix": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE121nnn/GSE121212/matrix/GSE121212_series_matrix.txt.gz",
        "counts": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE121nnn/GSE121212/suppl/GSE121212_readcount.txt.gz",
    },
    "GSE157194": {
        "type": "rnaseq",
        "matrix": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE157nnn/GSE157194/matrix/GSE157194_series_matrix.txt.gz",
        "counts": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE157nnn/GSE157194/suppl/GSE157194_Raw_gene_counts_matrix.txt.gz",
    },
}
GPL570_ANNOT_URL = "https://ftp.ncbi.nlm.nih.gov/geo/platforms/GPLnnn/GPL570/annot/GPL570.annot.gz"

def download_file_cached(url, dest_path, timeout=180):
    dest = Path(dest_path)
    if dest.exists() and dest.stat().st_size > 100:
        return dest
    r = SESSION.get(url, stream=True, timeout=timeout)
    r.raise_for_status()
    with open(dest, "wb") as fh:
        for chunk in r.iter_content(chunk_size=1024 * 256):
            if chunk:
                fh.write(chunk)
    return dest

def parse_geo_characteristics_kv(meta_df):
    """
    Plan 2: Parse !Sample_characteristics_ch1 (`char_*`) columns strictly into
    key-value columns (`kv_<key>`) so free-text `description` NEVER pollutes group matching.
    """
    char_cols = [c for c in meta_df.columns if c.startswith("char_")]
    kv_records = {gsm: {} for gsm in meta_df.index}
    for col in char_cols:
        for gsm, raw_val in meta_df[col].items():
            s = str(raw_val).strip()
            if ":" in s:
                k, v = s.split(":", 1)
                k_norm = re.sub(r"[^a-z0-9]+", "_", k.strip().lower()).strip("_")
                v_norm = v.strip()
                if k_norm:
                    if k_norm in kv_records[gsm] and v_norm and v_norm != kv_records[gsm][k_norm]:
                        kv_records[gsm][k_norm] = f"{kv_records[gsm][k_norm]} | {v_norm}"
                    else:
                        kv_records[gsm][k_norm] = v_norm
    kv_df = pd.DataFrame.from_dict(kv_records, orient="index").reindex(meta_df.index).fillna("")
    kv_df.columns = [f"kv_{c}" for c in kv_df.columns]
    return kv_df

def parse_geo_series_matrix(gz_path, load_expression=True):
    meta_rows = {}
    char_idx = 0
    expr_lines = []
    in_table = False

    opener = gzip.open if str(gz_path).endswith(".gz") else open
    with opener(gz_path, "rt", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            if line.startswith("!series_matrix_table_begin"):
                in_table = True
                continue
            if line.startswith("!series_matrix_table_end"):
                in_table = False
                continue
            if in_table:
                if load_expression:
                    expr_lines.append(line)
                continue
            if line.startswith("!Sample_"):
                parts = [p.strip().strip('"') for p in line.rstrip("\r\n").split("\t")]
                key = parts[0].replace("!Sample_", "")
                vals = parts[1:]
                if key == "characteristics_ch1":
                    key = f"char_{char_idx}"
                    char_idx += 1
                elif key in meta_rows:
                    key = f"{key}_{len(meta_rows)}"
                meta_rows[key] = vals

    if "geo_accession" not in meta_rows:
        raise ValueError(f"Could not find !Sample_geo_accession in {gz_path}")

    n_samples = len(meta_rows["geo_accession"])
    clean_meta = {k: vals for k, vals in meta_rows.items() if len(vals) == n_samples}
    meta_df = pd.DataFrame(clean_meta)
    meta_df.index = meta_df["geo_accession"].astype(str)

    # Structured KV columns from char_* only (NEVER description)
    kv_df = parse_geo_characteristics_kv(meta_df)
    meta_df = pd.concat([meta_df, kv_df], axis=1)

    # Clean char+title blob (excludes description!)
    safe_cols = [c for c in meta_df.columns if c == "title" or c.startswith("char_")]
    meta_df["safe_blob"] = meta_df[safe_cols].astype(str).agg(" | ".join, axis=1).str.lower()

    expr_df = pd.DataFrame()
    if load_expression and len(expr_lines) > 2:
        expr_df = pd.read_csv(io.StringIO("".join(expr_lines)), sep="\t", index_col=0, low_memory=False)
        expr_df.index = expr_df.index.astype(str).str.strip('"')
        expr_df.columns = [str(c).strip('"') for c in expr_df.columns]
        expr_df = expr_df.apply(pd.to_numeric, errors="coerce")

    return meta_df, expr_df

def _extract_kv_series(meta_df, key_patterns, fallback_col="title"):
    """Pick the first structured kv_* column matching any key_pattern; fallback to title (never description)."""
    kv_cols = [c for c in meta_df.columns if c.startswith("kv_")]
    for pat in key_patterns:
        for c in kv_cols:
            if re.search(pat, c[3:]):
                ser = meta_df[c].astype(str).str.strip()
                if (ser != "").sum() > 0:
                    return ser.str.lower(), c
    if fallback_col in meta_df.columns:
        return meta_df[fallback_col].astype(str).str.strip().str.lower(), fallback_col
    return pd.Series("", index=meta_df.index), "none"

def parse_gse130588_groups(meta_df):
    """
    Plan 2 Fix: Extract tissue, time, and treatment from structured char_* key-value fields
    (never scanning free-text description), and return diagnostic preview table.
    """
    tissue_s, tissue_col = _extract_kv_series(meta_df, [r"tissue", r"lesion", r"skin", r"disease_state", r"sample_type"])
    time_s, time_col = _extract_kv_series(meta_df, [r"time", r"week", r"visit", r"day"])
    trt_s, trt_col = _extract_kv_series(meta_df, [r"treatment", r"drug", r"arm", r"therapy", r"agent"])

    # Exact tissue matching (handles "ls", "lesional", "ad lesional", "nl", "non-lesional", "nonlesional")
    is_nl = tissue_s.str.contains(r"^nl$|\bnl\b|non[-_\s]?lesional|uninvolved", regex=True)
    is_ls = tissue_s.str.contains(r"^ls$|\bls\b|\blesional\b", regex=True) & ~is_nl
    is_w0 = time_s.str.contains(r"week[\s_-]*0\b|\bw0\b|baseline|day[\s_-]*1\b|^0$", regex=True)
    is_w4 = time_s.str.contains(r"week[\s_-]*4\b|\bw4\b|^4$", regex=True)
    is_w16 = time_s.str.contains(r"week[\s_-]*16\b|\bw16\b|^16$", regex=True)
    is_dupi = trt_s.str.contains(r"dupilumab|dupi", regex=True)

    # Secondary fallback to title token ONLY if structured column missed an arm
    if is_ls.sum() == 0 and "title" in meta_df.columns:
        t = meta_df["title"].astype(str).str.lower()
        is_nl = t.str.contains(r"[\s_-]nl(?:[\s_-]|$)|non[-_\s]?lesional", regex=True)
        is_ls = t.str.contains(r"[\s_-]ls(?:[\s_-]|$)|\blesional\b", regex=True) & ~is_nl
    if is_w0.sum() == 0 and "title" in meta_df.columns:
        t = meta_df["title"].astype(str).str.lower()
        is_w0 = t.str.contains(r"w0|wk0|week0|baseline", regex=True)
        is_w4 = t.str.contains(r"w4|wk4|week4", regex=True)
        is_w16 = t.str.contains(r"w16|wk16|week16", regex=True)
    if is_dupi.sum() == 0 and "title" in meta_df.columns:
        t = meta_df["title"].astype(str).str.lower()
        is_dupi = t.str.contains(r"dupi|dupilumab", regex=True)

    diag_df = pd.DataFrame({
        "title": meta_df.get("title", ""),
        f"tissue({tissue_col})": tissue_s,
        f"time({time_col})": time_s,
        f"treatment({trt_col})": trt_s,
        "is_LS": is_ls,
        "is_NL": is_nl,
        "is_Dupi": is_dupi,
    }, index=meta_df.index)

    groups = {
        "LS_W0": meta_df.index[is_ls & is_w0].tolist(),
        "NL_W0": meta_df.index[is_nl & is_w0].tolist(),
        "LS_Dupi_W0": meta_df.index[is_ls & is_dupi & is_w0].tolist(),
        "LS_Dupi_W4": meta_df.index[is_ls & is_dupi & is_w4].tolist(),
        "LS_Dupi_W16": meta_df.index[is_ls & is_dupi & is_w16].tolist(),
        "_diag_df": diag_df,
    }
    return groups

def parse_gse133477_groups(meta_df):
    tissue_s, _ = _extract_kv_series(meta_df, [r"tissue", r"lesion", r"skin", r"sample_type"])
    time_s, _ = _extract_kv_series(meta_df, [r"time", r"day", r"visit"])
    trt_s, _ = _extract_kv_series(meta_df, [r"treatment", r"drug", r"arm", r"agent"])
    blob = meta_df["safe_blob"]

    is_nl = tissue_s.str.contains(r"^nl$|\bnl\b|non[-_\s]?lesional", regex=True) | blob.str.contains(r"non[-_\s]?lesional|\bnl\b", regex=True)
    is_ls = (tissue_s.str.contains(r"^ls$|\bls\b|\blesional\b", regex=True) | blob.str.contains(r"\blesional\b|\bls\b", regex=True)) & ~is_nl
    is_d1 = time_s.str.contains(r"day[\s_-]*1\b|\bd1\b|baseline|^1$", regex=True) | blob.str.contains(r"day\s*1\b|\bd1\b|baseline", regex=True)
    is_d8 = time_s.str.contains(r"day[\s_-]*8\b|\bd8\b|^8$", regex=True) | blob.str.contains(r"day\s*8\b|\bd8\b", regex=True)
    is_d15 = time_s.str.contains(r"day[\s_-]*15\b|\bd15\b|^15$", regex=True) | blob.str.contains(r"day\s*15\b|\bd15\b", regex=True)
    is_crisa = trt_s.str.contains(r"crisaborole|crisa|an2728", regex=True) | blob.str.contains(r"crisaborole|crisa|an2728", regex=True)
    is_veh = trt_s.str.contains(r"vehicle|placebo", regex=True) | blob.str.contains(r"vehicle|placebo", regex=True)
    return {
        "LS_D1": meta_df.index[is_ls & is_d1].tolist(),
        "NL_D1": meta_df.index[is_nl & is_d1].tolist(),
        "LS_Crisa_D1": meta_df.index[is_ls & is_crisa & is_d1].tolist(),
        "LS_Crisa_D8": meta_df.index[is_ls & is_crisa & is_d8].tolist(),
        "LS_Crisa_D15": meta_df.index[is_ls & is_crisa & is_d15].tolist(),
        "LS_Veh_D15": meta_df.index[is_ls & is_veh & is_d15].tolist(),
    }

def parse_gse121212_groups(meta_df):
    blob = meta_df["safe_blob"]
    is_ad = blob.str.contains(r"atopic dermatitis|\bad\b", regex=True) & ~blob.str.contains(r"psoriasis|\bpso\b|healthy|control", regex=True)
    is_nl = blob.str.contains(r"non[- ]?lesional|uninvolved|\bnl\b", regex=True)
    is_ls = blob.str.contains(r"\blesional\b|\bls\b", regex=True) & ~is_nl
    return {
        "AD_LS": meta_df.index[is_ad & is_ls].tolist(),
        "AD_NL": meta_df.index[is_ad & is_nl].tolist(),
    }

def parse_gse157194_groups(meta_df):
    blob = meta_df["safe_blob"]
    is_m0 = blob.str.contains(r"\bm0\b|baseline", regex=True)
    is_m3 = blob.str.contains(r"\bm3\b|3\s*month|month\s*3|follow", regex=True)
    is_an = blob.str.contains(r"_an_|non[- ]?lesional|\ban\b", regex=True)
    is_al = blob.str.contains(r"_al_|\blesional\b|\bal\b", regex=True) & ~is_an
    is_dupi = blob.str.contains(r"dupilumab|dupi", regex=True)
    return {
        "AL_m0": meta_df.index[is_al & is_m0].tolist(),
        "AN_m0": meta_df.index[is_an & is_m0].tolist(),
        "AL_Dupi_m3": meta_df.index[is_al & is_dupi & is_m3].tolist(),
    }

def extract_scorad_from_meta(meta_df):
    kv_scorad, _ = _extract_kv_series(meta_df, [r"scorad"])
    scores = {}
    for gsm in meta_df.index:
        txt = str(kv_scorad.loc[gsm]) or str(meta_df.loc[gsm, "safe_blob"])
        m = re.search(r"([0-9]+(?:\.[0-9]+)?)", txt) if kv_scorad.loc[gsm] else re.search(r"scorad[^0-9.-]*([0-9]+(?:\.[0-9]+)?)", txt)
        if m:
            scores[gsm] = float(m.group(1))
    return pd.Series(scores, dtype=float)

def load_gpl570_probe_map(cache_path="data/GPL570.annot.gz"):
    p = download_file_cached(GPL570_ANNOT_URL, cache_path, timeout=180)
    probe_to_sym = {}
    in_table = False
    header = None
    with gzip.open(p, "rt", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            if line.startswith("!platform_table_begin"):
                in_table = True
                continue
            if line.startswith("!platform_table_end"):
                break
            if not in_table:
                continue
            parts = line.rstrip("\r\n").split("\t")
            if header is None:
                header = [h.strip().lower() for h in parts]
                id_idx = header.index("id") if "id" in header else 0
                sym_idx = next((i for i, h in enumerate(header) if "gene symbol" in h or h == "symbol"), 2)
                continue
            if len(parts) > max(id_idx, sym_idx):
                pid = parts[id_idx].strip()
                raw_sym = parts[sym_idx].strip()
                if pid and raw_sym and raw_sym != "---":
                    first_sym = raw_sym.split("///")[0].strip()
                    if first_sym:
                        probe_to_sym[pid] = normalize_symbol(first_sym)
    return probe_to_sym

def map_ensembl_to_symbol(ensembl_ids):
    clean_ids = sorted({str(e).split(".")[0] for e in ensembl_ids if pd.notna(e) and str(e).startswith("ENSG")})
    mapping = {}
    for i in range(0, len(clean_ids), 1000):
        chunk = clean_ids[i:i + 1000]
        try:
            r = SESSION.post("https://mygene.info/v3/gene", json={"ids": chunk, "fields": "symbol"}, timeout=60)
            if r.ok:
                for item in r.json():
                    if isinstance(item, dict) and "symbol" in item and "query" in item:
                        mapping[item["query"]] = normalize_symbol(item["symbol"])
        except Exception as e:
            print(f"    [WARN] MyGene chunk {i} failed: {e}")
    return mapping

def collapse_probes_to_genes(expr_df, probe_map):
    df = expr_df.copy()
    df["__symbol__"] = df.index.map(probe_map)
    df = df.dropna(subset=["__symbol__"])
    num_cols = [c for c in df.columns if c != "__symbol__"]
    df["__var__"] = df[num_cols].var(axis=1, skipna=True)
    df = df.sort_values("__var__", ascending=False).drop_duplicates(subset=["__symbol__"], keep="first")
    df = df.set_index("__symbol__")[num_cols]
    q99 = np.nanpercentile(df.to_numpy(dtype=float), 99)
    if q99 > 50:
        df = np.log2(np.clip(df, a_min=0, a_max=None) + 1.0)
    return df

def counts_to_log2cpm(counts_df):
    mat = counts_df.apply(pd.to_numeric, errors="coerce").fillna(0.0)
    lib_sizes = mat.sum(axis=0).replace(0, np.nan)
    cpm = mat.div(lib_sizes, axis=1) * 1e6
    return np.log2(cpm + 1.0)

def benjamini_hochberg(pvals):
    p = np.asarray(pvals, dtype=float)
    n = len(p)
    q = np.full(n, np.nan, dtype=float)
    valid = np.isfinite(p)
    if not np.any(valid):
        return q
    pv = p[valid]
    order = np.argsort(pv)
    ranked = pv[order]
    m = len(ranked)
    adj = ranked * m / np.arange(1, m + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.clip(adj, 0.0, 1.0)
    out = np.empty(m, dtype=float)
    out[order] = adj
    q[valid] = out
    return q

def compute_deg_table(expr_df, case_cols, ctrl_cols, min_n=3):
    case_cols = [c for c in case_cols if c in expr_df.columns]
    ctrl_cols = [c for c in ctrl_cols if c in expr_df.columns]
    if len(case_cols) < min_n or len(ctrl_cols) < min_n:
        return pd.DataFrame(columns=["symbol", "log2FC", "t_stat", "pval", "padj", "n_case", "n_ctrl"])
    x1 = expr_df[case_cols].to_numpy(dtype=float)
    x0 = expr_df[ctrl_cols].to_numpy(dtype=float)
    log2fc = np.nanmean(x1, axis=1) - np.nanmean(x0, axis=1)
    t_stat, pval = ttest_ind(x1, x0, axis=1, equal_var=False, nan_policy="omit")
    padj = benjamini_hochberg(pval)
    out = pd.DataFrame({
        "symbol": expr_df.index.astype(str),
        "log2FC": log2fc, "t_stat": t_stat, "pval": pval, "padj": padj,
        "n_case": len(case_cols), "n_ctrl": len(ctrl_cols),
    })
    out["symbol"] = out["symbol"].map(normalize_symbol)
    out = out.replace([np.inf, -np.inf], np.nan).dropna(subset=["symbol", "log2FC", "t_stat"])
    return out.groupby("symbol", as_index=False).agg(
        log2FC=("log2FC", "mean"), t_stat=("t_stat", "mean"),
        pval=("pval", "min"), padj=("padj", "min"),
        n_case=("n_case", "first"), n_ctrl=("n_ctrl", "first"),
    )

def match_rnaseq_columns_to_meta(counts_cols, meta_df):
    col_to_gsm = {}
    gsm_set = set(meta_df.index.astype(str))
    title_map, desc_map = {}, {}
    for gsm, row in meta_df.iterrows():
        t = str(row.get("title", "")).strip()
        if t:
            title_map[t.lower()] = gsm
            title_map[re.sub(r"[^a-z0-9]+", "_", t.lower()).strip("_")] = gsm
        for c in meta_df.columns:
            if c.startswith("description"):
                d = str(row[c]).strip()
                if d:
                    desc_map[d.lower()] = gsm
                    desc_map[re.sub(r"[^a-z0-9]+", "_", d.lower()).strip("_")] = gsm
    for col in counts_cols:
        cs = str(col).strip()
        cl = cs.lower()
        cnorm = re.sub(r"[^a-z0-9]+", "_", cl).strip("_")
        if cs in gsm_set:
            col_to_gsm[col] = cs
        elif cl in title_map:
            col_to_gsm[col] = title_map[cl]
        elif cnorm in title_map:
            col_to_gsm[col] = title_map[cnorm]
        elif cl in desc_map:
            col_to_gsm[col] = desc_map[cl]
        elif cnorm in desc_map:
            col_to_gsm[col] = desc_map[cnorm]
        else:
            for k, gsm in title_map.items():
                if len(k) >= 5 and (k in cl or cl in k):
                    col_to_gsm[col] = gsm
                    break
    return col_to_gsm


# =============================================================================
# 5. VECTORIZED 2,000-PERMUTATION WTCS / GSEA ENGINE (PLAN 4)
# =============================================================================
def gsea_es_weighted(ranked_series, gene_set, weight=1.0):
    s = ranked_series.dropna().sort_values(ascending=False)
    genes = s.index.to_numpy()
    vals = s.to_numpy(dtype=float)
    n_total = len(genes)
    if n_total == 0:
        return np.nan, 0
    gset = set(gene_set)
    hits = np.array([g in gset for g in genes], dtype=bool)
    n_hits = int(hits.sum())
    n_miss = n_total - n_hits
    if n_hits < 2 or n_miss == 0:
        return np.nan, n_hits
    hit_weights = np.abs(vals) ** weight if weight > 0 else np.ones_like(vals)
    nr = float(hit_weights[hits].sum())
    if nr == 0:
        return np.nan, n_hits
    running = np.cumsum(np.where(hits, hit_weights / nr, -1.0 / n_miss))
    return float(running[int(np.argmax(np.abs(running)))]), n_hits

def _vectorized_null_es(abs_vals, k, n_perm, rng, batch_size=250):
    """Fast vectorized O(B * k log k) weighted GSEA ES null distribution for gene set size k."""
    n_tot = len(abs_vals)
    if k < 2 or k >= n_tot:
        return np.full(n_perm, np.nan, dtype=float)
    n_miss = float(n_tot - k)
    out = np.empty(n_perm, dtype=float)
    pos = 0
    while pos < n_perm:
        b = min(batch_size, n_perm - pos)
        # Draw b independent random subsets of size k from [0, n_tot)
        idx = np.empty((b, k), dtype=np.int32)
        for i in range(b):
            idx[i] = np.sort(rng.choice(n_tot, size=k, replace=False))
        w = abs_vals[idx]                              # (b, k)
        nr = w.sum(axis=1, keepdims=True)
        nr = np.where(nr == 0, 1.0, nr)
        p_hit_at = np.cumsum(w / nr, axis=1)           # cumulative hit score right AFTER each hit
        p_hit_before = p_hit_at - (w / nr)             # cumulative hit score right BEFORE each hit
        # Number of misses strictly before hit j is (idx[:, j] - j)
        misses_before = idx - np.arange(k, dtype=np.int32)[None, :]
        p_miss_at = misses_before / n_miss
        # Extrema of running sum only occur immediately before or after a hit!
        peak_pos = np.max(p_hit_at - p_miss_at, axis=1)
        peak_neg = np.min(p_hit_before - p_miss_at, axis=1)
        out[pos:pos + b] = np.where(np.abs(peak_pos) >= np.abs(peak_neg), peak_pos, peak_neg)
        pos += b
    return out

def compute_wtcs(drug_vec, up_genes, down_genes, n_perm=GSEA_PERMUTATIONS, seed=RANDOM_SEED):
    es_up, n_up = gsea_es_weighted(drug_vec, up_genes)
    es_dn, n_dn = gsea_es_weighted(drug_vec, down_genes)
    if pd.isna(es_up) and pd.isna(es_dn):
        return {"wtcs": np.nan, "raw_cs": np.nan, "es_up": np.nan, "es_down": np.nan, "n_up": n_up, "n_down": n_dn, "pval": np.nan}

    if pd.isna(es_up) and pd.notna(es_dn):
        raw_cs = -float(es_dn)
        wtcs = raw_cs
    elif pd.notna(es_up) and pd.isna(es_dn):
        raw_cs = float(es_up)
        wtcs = raw_cs
    else:
        raw_cs = 0.5 * (float(es_up) - float(es_dn))
        wtcs = raw_cs if (np.sign(es_up) != np.sign(es_dn)) else 0.0

    pval = np.nan
    if n_perm > 0 and pd.notna(raw_cs):
        rng = np.random.default_rng(seed)
        abs_vals = np.abs(drug_vec.dropna().sort_values(ascending=False).to_numpy(dtype=float))
        null_u = _vectorized_null_es(abs_vals, n_up, n_perm, rng) if pd.notna(es_up) else None
        null_d = _vectorized_null_es(abs_vals, n_dn, n_perm, rng) if pd.notna(es_dn) else None
        if null_u is not None and null_d is not None:
            null_arr = 0.5 * (null_u - null_d)
        elif null_u is not None:
            null_arr = null_u
        else:
            null_arr = -null_d
        valid = np.isfinite(null_arr)
        if np.any(valid):
            pval = float((np.sum(np.abs(null_arr[valid]) >= abs(raw_cs)) + 1) / (int(valid.sum()) + 1))

    return {
        "wtcs": round(float(wtcs), 4),
        "raw_cs": round(float(raw_cs), 4),
        "es_up": round(float(es_up), 4) if pd.notna(es_up) else np.nan,
        "es_down": round(float(es_dn), 4) if pd.notna(es_dn) else np.nan,
        "n_up": int(n_up),
        "n_down": int(n_dn),
        "pval": round(float(pval), 5) if pd.notna(pval) else np.nan,
    }

def safe_cosine_spearman(s1, s2, gene_subset=None, min_n=5):
    if gene_subset is not None:
        glist = list(gene_subset)
        s1 = s1.reindex(glist)
        s2 = s2.reindex(glist)
    pair = pd.concat([s1.rename("a"), s2.rename("b")], axis=1).dropna()
    n = len(pair)
    if n < min_n:
        return np.nan, np.nan, np.nan, n
    v1 = pair["a"].to_numpy(dtype=float)
    v2 = pair["b"].to_numpy(dtype=float)
    if np.all(v1 == 0) or np.all(v2 == 0):
        return np.nan, np.nan, np.nan, n
    cos_sim = 1.0 - cosine(v1, v2)
    rho, pval = spearmanr(v1, v2)
    return round(float(cos_sim), 4), round(float(rho), 4), round(float(pval), 5), int(n)


# =============================================================================
# 6. MAIN END-TO-END STUDY 1 (v2 ENHANCED) EXECUTION
# =============================================================================
def run_study1_pipeline():
    qc_flags = []
    t_start = time.time()
    print("=" * 92)
    print("STUDY 1 (v2 ENHANCED): DOSE- & TISSUE-STRATIFIED MONTELUKAST L1000 vs. AD & BENCHMARKS")
    print("=" * 92)

    # -------------------------------------------------------------------------
    # STEP 1: Standardized GO:BP + KEGG + Reactome Pathways (Plan 3)
    # -------------------------------------------------------------------------
    print("\n[STEP 1/6] Resolving standardized GO:BP + KEGG + Reactome pathways...")
    ad_submodules_norm, indiv_pathways, pathway_inv_df, gene_prov_df = build_standardized_ad_submodules(AD_PATHWAY_SPECS)
    pathway_inv_df.to_csv("results/results_standardized_pathway_inventory.csv", index=False)
    gene_prov_df.to_csv("results/results_ad_submodule_genes.csv", index=False)

    failed_pw = pathway_inv_df[pathway_inv_df["n_genes"] == 0]["pathway_id"].tolist()
    if failed_pw:
        qc_flags.append(f"PATHWAY RETRIEVAL GAPS: {len(failed_pw)} term(s) returned 0 genes -> {', '.join(failed_pw)}")

    # -------------------------------------------------------------------------
    # STEP 2: LINCS L1000 Union Retrieval + Dose & Tissue Stratification (Plans 1 & 4)
    # -------------------------------------------------------------------------
    print("\n[STEP 2/6] Querying SigCom LINCS L1000 (union pert_name + pert_id + smiles + CysLT1R proxies)...")
    l1000_vectors = {}
    inventory_rows = []
    montelukast_dose_df = pd.DataFrame()
    dose_fallback_notes = []

    def _register_stratum(label, role, s_list, tissue_tag="PanCell"):
        vec, used_meta, n_used = collapse_signatures_to_consensus(s_list)
        cells = sorted({str((m.get("meta") or {}).get("cell_line", "")).upper() for m in used_meta if (m.get("meta") or {}).get("cell_line")})
        inventory_rows.append({
            "signature": label, "role": role, "tissue_stratum": tissue_tag,
            "n_found": len(s_list), "n_used": n_used, "n_genes": len(vec),
            "n_cell_lines": len(cells), "sample_cells": ",".join(cells[:10]),
        })
        if not vec.empty:
            l1000_vectors[label] = vec
            print(f"      -> {label:32s}: {len(vec):,} genes | {n_used}/{len(s_list)} sigs ({len(cells)} cells)")
        return vec, n_used

    for name, spec in PERTURBAGENS.items():
        sigs = query_sigcom_signatures(spec)
        print(f"  - {name:12s} ({spec['role']}): {len(sigs)} union signatures found")
        if not sigs:
            inventory_rows.append({
                "signature": name, "role": spec["role"], "tissue_stratum": "PanCell",
                "n_found": 0, "n_used": 0, "n_genes": 0, "n_cell_lines": 0, "sample_cells": "",
            })
            qc_flags.append(f"MISSING L1000 PERTURBAGEN: {name} ({spec['role']}) returned 0 signatures.")
            continue

        if spec.get("stratify_dose"):
            bins = {"Low_le_1uM": [], "Mid_1_to_10uM": [], "High_ge_10uM": [], "Unknown_Dose": []}
            rec_rows = []
            for s in sigs:
                meta = s.get("meta") or {}
                d_um = parse_dose_um(meta)
                b = classify_dose_bin(d_um)
                cell = str(meta.get("cell_line", "")).upper()
                bins[b].append(s)
                rec_rows.append({
                    "compound": name, "sig_id": s.get("id", ""), "pert_name": meta.get("pert_name", ""),
                    "pert_id": meta.get("pert_id", ""), "cell_line": cell,
                    "tissue_stratum": classify_cell_tissue(cell),
                    "pert_dose_raw": meta.get("pert_dose", ""), "pert_dose_unit": meta.get("pert_dose_unit", ""),
                    "dose_uM": d_um, "dose_bin": b, "pert_time": meta.get("pert_time", ""),
                })
            rec_df = pd.DataFrame(rec_rows)
            rec_df.to_csv(f"results/{name.lower()}_l1000_dose_records.csv", index=False)
            if name == "Montelukast":
                montelukast_dose_df = rec_df

            if name == "Montelukast":
                strata = [
                    ("Montelukast_Low", bins["Low_le_1uM"]),
                    ("Montelukast_Mid", bins["Mid_1_to_10uM"]),
                    ("Montelukast_High", bins["High_ge_10uM"]),
                    ("Montelukast_All", sigs),
                ]
            else:
                strata = [
                    (f"{name}_Low", bins["Low_le_1uM"]),
                    (f"{name}_All", sigs),
                ]

            for s_lbl, s_list in strata:
                _, n_u = _register_stratum(s_lbl, spec["role"], s_list, "PanCell")
                if name == "Montelukast" and s_lbl == "Montelukast_Low" and n_u == 0:
                    msg = (
                        "PLAN 1 FALLBACK TRIGGERED: Montelukast <=1 uM still had 0 SigCom LINCS signatures "
                        "after unioning pert_name + BRD pert_ids + SMILES; 2-tier CysLT1R on-target proxies "
                        "(Zafirlukast_Low/All, Pranlukast_Low/All, CYSLTR1_KD) are active for low-dose comparison."
                    )
                    qc_flags.append(msg)
                    dose_fallback_notes.append(msg)

            # Plan 4: Tissue-relevance stratification on Montelukast_High (and Low if >=2 sigs)
            if name == "Montelukast":
                for d_lbl, d_list in [("Montelukast_High", bins["High_ge_10uM"]), ("Montelukast_Low", bins["Low_le_1uM"])]:
                    rel = [s for s in d_list if classify_cell_tissue((s.get("meta") or {}).get("cell_line")) == "RelevantTissue"]
                    hor = [s for s in d_list if classify_cell_tissue((s.get("meta") or {}).get("cell_line")) == "HormoneTumor"]
                    if len(rel) >= 2:
                        _register_stratum(f"{d_lbl}__RelevantTissue", f"{spec['role']}_TissueStrat", rel, "RelevantTissue")
                    if len(hor) >= 2:
                        _register_stratum(f"{d_lbl}__HormoneTumor", f"{spec['role']}_TissueStrat", hor, "HormoneTumor")

                # Matched-cell Low vs High sensitivity control
                low_cells = {str((s.get("meta") or {}).get("cell_line", "")).upper() for s in bins["Low_le_1uM"]}
                high_cells = {str((s.get("meta") or {}).get("cell_line", "")).upper() for s in bins["High_ge_10uM"]}
                shared_cells = sorted((low_cells & high_cells) - {"", "NONE"})
                if shared_cells:
                    low_sh = [s for s in bins["Low_le_1uM"] if str((s.get("meta") or {}).get("cell_line", "")).upper() in shared_cells]
                    hi_sh = [s for s in bins["High_ge_10uM"] if str((s.get("meta") or {}).get("cell_line", "")).upper() in shared_cells]
                    _register_stratum("Montelukast_Low_MatchedCells", "Sensitivity_MatchedCells", low_sh, "MatchedCells")
                    _register_stratum("Montelukast_High_MatchedCells", "Sensitivity_MatchedCells", hi_sh, "MatchedCells")
        else:
            _register_stratum(name, spec["role"], sigs, "PanCell")
            # Plan 4: Tissue-relevance stratification on mechanistic controls & CYSLTR1_KD
            rel = [s for s in sigs if classify_cell_tissue((s.get("meta") or {}).get("cell_line")) == "RelevantTissue"]
            hor = [s for s in sigs if classify_cell_tissue((s.get("meta") or {}).get("cell_line")) == "HormoneTumor"]
            if len(rel) >= 2:
                _register_stratum(f"{name}__RelevantTissue", f"{spec['role']}_TissueStrat", rel, "RelevantTissue")
            if len(hor) >= 2:
                _register_stratum(f"{name}__HormoneTumor", f"{spec['role']}_TissueStrat", hor, "HormoneTumor")
        time.sleep(0.3)

    inv_df = pd.DataFrame(inventory_rows)
    inv_df.to_csv("results/results_l1000_signature_inventory.csv", index=False)
    l1000_df = pd.DataFrame(l1000_vectors).dropna(how="all")
    l1000_df.to_csv("results/l1000_consensus_matrix.csv.gz", compression="gzip")
    if l1000_df.empty:
        raise RuntimeError("No LINCS L1000 signatures retrieved.")

    # -------------------------------------------------------------------------
    # STEP 3: Download & Process GEO AD Cohorts + Clinical Benchmarks (Plan 2)
    # -------------------------------------------------------------------------
    print("\n[STEP 3/6] Downloading & processing GEO AD cohorts (structured metadata parsing)...")
    probe_map = None
    ad_disease_degs = {}
    clinical_reversal_degs = {}
    geo_audit_rows = []
    scorad_corr_summary = []
    gse130588_diag_preview = pd.DataFrame()

    for gse in ["GSE130588", "GSE133477"]:
        try:
            if probe_map is None:
                print("  - Loading GPL570 probe-to-symbol annotation...")
                probe_map = load_gpl570_probe_map()
            mat_path = download_file_cached(GEO_URLS[gse]["matrix"], f"data/{gse}_series_matrix.txt.gz")
            meta_df, raw_expr = parse_geo_series_matrix(mat_path, load_expression=True)
            gene_expr = collapse_probes_to_genes(raw_expr, probe_map)

            if gse == "GSE130588":
                grp = parse_gse130588_groups(meta_df)
                gse130588_diag_preview = grp["_diag_df"]
                gse130588_diag_preview.to_csv("results/gse130588_parsed_sample_metadata.csv")
                print("  - GSE130588 parsed metadata preview (first 3 rows):")
                print(gse130588_diag_preview.head(3).to_string())

                # Plan 2: Strict arm-size verification
                for arm_k in ["LS_W0", "NL_W0", "LS_Dupi_W0", "LS_Dupi_W16"]:
                    if len(grp[arm_k]) < 5:
                        qc_flags.append(f"GSE130588 ARM WARNING: {arm_k} has only n={len(grp[arm_k])} samples (<5). Inspect results/gse130588_parsed_sample_metadata.csv.")

                deg_ad = compute_deg_table(gene_expr, grp["LS_W0"], grp["NL_W0"])
                if not deg_ad.empty:
                    ad_disease_degs[gse] = deg_ad
                    deg_ad.to_csv(f"results/deg_AD_LS_vs_NL_{gse}.csv", index=False)
                for lbl, post_key in [("Dupilumab_W16_vs_W0_GSE130588", "LS_Dupi_W16"), ("Dupilumab_W4_vs_W0_GSE130588", "LS_Dupi_W4")]:
                    d_cl = compute_deg_table(gene_expr, grp[post_key], grp["LS_Dupi_W0"])
                    if not d_cl.empty:
                        clinical_reversal_degs[lbl] = d_cl
                        d_cl.to_csv(f"results/deg_clinical_{lbl}.csv", index=False)

                scorad_s = extract_scorad_from_meta(meta_df)
                if len(scorad_s) >= 10:
                    common_s = scorad_s.index.intersection(gene_expr.columns)
                    for mod_name, mod_genes in ad_submodules_norm.items():
                        ov = gene_expr.index.intersection(mod_genes)
                        if len(ov) >= 3 and len(common_s) >= 10:
                            mod_score = gene_expr.loc[ov, common_s].mean(axis=0)
                            rho, pval = spearmanr(mod_score.values, scorad_s.loc[common_s].values)
                            scorad_corr_summary.append({
                                "cohort": gse, "sub_module": mod_name, "n_samples": len(common_s),
                                "n_genes": len(ov), "scorad_spearman_rho": round(float(rho), 4), "pval": round(float(pval), 5),
                            })
                geo_audit_rows.append({
                    "cohort": gse, "status": "OK", "genes_mapped": len(gene_expr),
                    "LS_n": len(grp["LS_W0"]), "NL_n": len(grp["NL_W0"]),
                    "clinical_arms": f"Dupi_W0={len(grp['LS_Dupi_W0'])}, W4={len(grp['LS_Dupi_W4'])}, W16={len(grp['LS_Dupi_W16'])}",
                })
                print(f"  [OK] {gse}: {len(gene_expr):,} genes | LS_W0={len(grp['LS_W0'])}, NL_W0={len(grp['NL_W0'])}, Dupi_W0={len(grp['LS_Dupi_W0'])}, W4={len(grp['LS_Dupi_W4'])}, W16={len(grp['LS_Dupi_W16'])}")

            elif gse == "GSE133477":
                grp = parse_gse133477_groups(meta_df)
                deg_ad = compute_deg_table(gene_expr, grp["LS_D1"], grp["NL_D1"])
                if not deg_ad.empty:
                    ad_disease_degs[gse] = deg_ad
                    deg_ad.to_csv(f"results/deg_AD_LS_vs_NL_{gse}.csv", index=False)
                base_crisa = grp["LS_Crisa_D1"] if len(grp["LS_Crisa_D1"]) >= 3 else grp["LS_D1"]
                for lbl, post_key in [("Crisaborole_D15_vs_D1_GSE133477", "LS_Crisa_D15"), ("Crisaborole_D8_vs_D1_GSE133477", "LS_Crisa_D8")]:
                    d_cl = compute_deg_table(gene_expr, grp[post_key], base_crisa)
                    if not d_cl.empty:
                        clinical_reversal_degs[lbl] = d_cl
                        d_cl.to_csv(f"results/deg_clinical_{lbl}.csv", index=False)
                if len(grp["LS_Crisa_D15"]) >= 3 and len(grp["LS_Veh_D15"]) >= 3:
                    d_veh = compute_deg_table(gene_expr, grp["LS_Crisa_D15"], grp["LS_Veh_D15"])
                    if not d_veh.empty:
                        clinical_reversal_degs["Crisaborole_vs_Vehicle_D15_GSE133477"] = d_veh
                        d_veh.to_csv("results/deg_clinical_Crisaborole_vs_Vehicle_D15_GSE133477.csv", index=False)
                geo_audit_rows.append({
                    "cohort": gse, "status": "OK", "genes_mapped": len(gene_expr),
                    "LS_n": len(grp["LS_D1"]), "NL_n": len(grp["NL_D1"]),
                    "clinical_arms": f"Crisa_D1={len(base_crisa)}, D8={len(grp['LS_Crisa_D8'])}, D15={len(grp['LS_Crisa_D15'])}, Veh_D15={len(grp['LS_Veh_D15'])}",
                })
                print(f"  [OK] {gse}: {len(gene_expr):,} genes | LS_D1={len(grp['LS_D1'])}, NL_D1={len(grp['NL_D1'])}, Crisa_D15={len(grp['LS_Crisa_D15'])}")
        except Exception as e:
            qc_flags.append(f"GEO LOAD FAILED ({gse}): {e}")
            geo_audit_rows.append({"cohort": gse, "status": f"ERROR: {e}", "genes_mapped": 0, "LS_n": 0, "NL_n": 0, "clinical_arms": ""})
            print(f"  [WARN] {gse} failed: {e}")

    for gse in ["GSE121212", "GSE157194"]:
        try:
            mat_path = download_file_cached(GEO_URLS[gse]["matrix"], f"data/{gse}_series_matrix.txt.gz")
            meta_df, _ = parse_geo_series_matrix(mat_path, load_expression=False)
            cnt_path = download_file_cached(GEO_URLS[gse]["counts"], f"data/{gse}_counts.txt.gz")
            counts = pd.read_csv(cnt_path, sep="\t", index_col=0, low_memory=False)
            if counts.shape[1] < 5:
                counts = pd.read_csv(cnt_path, sep=r"\s+|,", engine="python", index_col=0)

            idx_str = counts.index.astype(str)
            if idx_str.str.startswith("ENSG").mean() > 0.4:
                ens_map = map_ensembl_to_symbol(idx_str)
                counts["__sym__"] = idx_str.str.split(".").str[0].map(ens_map)
                counts = counts.dropna(subset=["__sym__"]).groupby("__sym__").sum(numeric_only=True)
            else:
                counts.index = idx_str.map(normalize_symbol)
                counts = counts.groupby(level=0).sum(numeric_only=True)

            col_to_gsm = match_rnaseq_columns_to_meta(counts.columns, meta_df)
            counts = counts[[c for c in counts.columns if c in col_to_gsm]].rename(columns=col_to_gsm)
            counts = counts.loc[:, ~counts.columns.duplicated()]
            log_cpm = counts_to_log2cpm(counts)

            if gse == "GSE121212":
                grp = parse_gse121212_groups(meta_df)
                deg_ad = compute_deg_table(log_cpm, grp["AD_LS"], grp["AD_NL"])
                if not deg_ad.empty:
                    ad_disease_degs[gse] = deg_ad
                    deg_ad.to_csv(f"results/deg_AD_LS_vs_NL_{gse}.csv", index=False)
                geo_audit_rows.append({
                    "cohort": gse, "status": "OK", "genes_mapped": len(log_cpm),
                    "LS_n": len(set(grp["AD_LS"]) & set(log_cpm.columns)),
                    "NL_n": len(set(grp["AD_NL"]) & set(log_cpm.columns)),
                    "clinical_arms": "AD RNA-seq disease cohort",
                })
                print(f"  [OK] {gse}: {len(log_cpm):,} genes | AD_LS={len(set(grp['AD_LS']) & set(log_cpm.columns))}, AD_NL={len(set(grp['AD_NL']) & set(log_cpm.columns))}")

            elif gse == "GSE157194":
                grp = parse_gse157194_groups(meta_df)
                deg_ad = compute_deg_table(log_cpm, grp["AL_m0"], grp["AN_m0"])
                if not deg_ad.empty:
                    ad_disease_degs[gse] = deg_ad
                    deg_ad.to_csv(f"results/deg_AD_LS_vs_NL_{gse}.csv", index=False)
                d_dupi_rna = compute_deg_table(log_cpm, grp["AL_Dupi_m3"], grp["AL_m0"])
                if not d_dupi_rna.empty:
                    clinical_reversal_degs["Dupilumab_m3_vs_m0_GSE157194"] = d_dupi_rna
                    d_dupi_rna.to_csv("results/deg_clinical_Dupilumab_m3_vs_m0_GSE157194.csv", index=False)
                geo_audit_rows.append({
                    "cohort": gse, "status": "OK", "genes_mapped": len(log_cpm),
                    "LS_n": len(set(grp["AL_m0"]) & set(log_cpm.columns)),
                    "NL_n": len(set(grp["AN_m0"]) & set(log_cpm.columns)),
                    "clinical_arms": f"Dupi_m3={len(set(grp['AL_Dupi_m3']) & set(log_cpm.columns))}",
                })
                print(f"  [OK] {gse}: {len(log_cpm):,} genes | AL_m0={len(set(grp['AL_m0']) & set(log_cpm.columns))}, AN_m0={len(set(grp['AN_m0']) & set(log_cpm.columns))}")
        except Exception as e:
            qc_flags.append(f"GEO LOAD FAILED ({gse}): {e}")
            geo_audit_rows.append({"cohort": gse, "status": f"ERROR: {e}", "genes_mapped": 0, "LS_n": 0, "NL_n": 0, "clinical_arms": ""})
            print(f"  [WARN] {gse} failed: {e}")

    geo_audit_df = pd.DataFrame(geo_audit_rows)
    geo_audit_df.to_csv("results/results_geo_cohort_inventory.csv", index=False)
    if scorad_corr_summary:
        pd.DataFrame(scorad_corr_summary).to_csv("results/results_gse130588_scorad_module_corr.csv", index=False)

    if not ad_disease_degs:
        raise RuntimeError("No GEO AD disease cohorts loaded successfully.")

    # -------------------------------------------------------------------------
    # STEP 4: Consensus AD Signature & Standardized Sub-Module Splits
    # -------------------------------------------------------------------------
    print("\n[STEP 4/6] Computing cross-cohort consensus AD signature & standardized module splits...")
    fc_mat = pd.DataFrame({gse: df.set_index("symbol")["log2FC"] for gse, df in ad_disease_degs.items()})
    t_mat = pd.DataFrame({gse: df.set_index("symbol")["t_stat"] for gse, df in ad_disease_degs.items()})
    consensus_ad = pd.DataFrame({
        "n_cohorts": fc_mat.notna().sum(axis=1),
        "mean_log2FC": fc_mat.mean(axis=1),
        "median_log2FC": fc_mat.median(axis=1),
        "mean_t_stat": t_mat.mean(axis=1),
        "sign_concordance": fc_mat.apply(lambda r: abs(np.sign(r.dropna()).mean()) if r.notna().any() else np.nan, axis=1),
    })
    consensus_ad.index.name = "symbol"
    min_cohorts_req = 2 if len(ad_disease_degs) >= 2 else 1
    consensus_filt = consensus_ad[consensus_ad["n_cohorts"] >= min_cohorts_req].copy()
    consensus_filt.to_csv("results/results_consensus_AD_signature.csv")

    l1000_genes = set(l1000_df.index)
    cons_l1000 = consensus_filt.loc[consensus_filt.index.intersection(l1000_genes)]
    ad_up_global = cons_l1000[cons_l1000["mean_log2FC"] > 0].sort_values("mean_t_stat", ascending=False).head(TOP_N_DEG_GSEA).index.tolist()
    ad_dn_global = cons_l1000[cons_l1000["mean_log2FC"] < 0].sort_values("mean_t_stat", ascending=True).head(TOP_N_DEG_GSEA).index.tolist()

    module_sets = {
        "Global_AD_Consensus": {"up": ad_up_global, "down": ad_dn_global, "all": ad_up_global + ad_dn_global},
    }
    for mod_name, genes in ad_submodules_norm.items():
        if not genes:
            continue
        sub_c = consensus_filt.reindex(genes).dropna(subset=["mean_log2FC"])
        module_sets[mod_name] = {
            "up": sub_c[sub_c["mean_log2FC"] > 0].index.tolist(),
            "down": sub_c[sub_c["mean_log2FC"] < 0].index.tolist(),
            "all": list(genes),
        }

    indiv_pathway_sets = {}
    if SCORE_INDIVIDUAL_PATHWAYS:
        for pw_lbl, genes in indiv_pathways.items():
            sub_c = consensus_filt.reindex(genes).dropna(subset=["mean_log2FC"])
            indiv_pathway_sets[pw_lbl] = {
                "up": sub_c[sub_c["mean_log2FC"] > 0].index.tolist(),
                "down": sub_c[sub_c["mean_log2FC"] < 0].index.tolist(),
                "all": list(genes),
            }

    # -------------------------------------------------------------------------
    # STEP 5: Vectorized 2,000-Perm Reversal Scoring + BH-FDR (Plan 4)
    # -------------------------------------------------------------------------
    print(f"\n[STEP 5/6] Running vectorized {GSEA_PERMUTATIONS}-permutation WTCS/GSEA + BH-FDR...")
    ad_fc_vec = consensus_filt["mean_log2FC"]

    def _score_set_collection(set_dict):
        rows = []
        for mod_name, m_dict in set_dict.items():
            for sig_name in l1000_df.columns:
                drug_vec = l1000_df[sig_name].dropna()
                wtcs_res = compute_wtcs(drug_vec, m_dict["up"], m_dict["down"], n_perm=GSEA_PERMUTATIONS, seed=RANDOM_SEED)
                cos_v, rho_v, sp_p, n_ov = safe_cosine_spearman(drug_vec, ad_fc_vec, gene_subset=m_dict["all"], min_n=4)
                rows.append({
                    "module": mod_name,
                    "perturbagen": sig_name,
                    "wtcs": wtcs_res["wtcs"],
                    "raw_cs": wtcs_res["raw_cs"],
                    "es_up": wtcs_res["es_up"],
                    "es_down": wtcs_res["es_down"],
                    "wtcs_pval": wtcs_res["pval"],
                    "n_up_l1000": wtcs_res["n_up"],
                    "n_down_l1000": wtcs_res["n_down"],
                    "reversal_cosine": cos_v,
                    "reversal_spearman": rho_v,
                    "spearman_pval": sp_p,
                    "n_shared": n_ov,
                })
        df = pd.DataFrame(rows)
        if not df.empty:
            df["wtcs_fdr"] = np.round(benjamini_hochberg(df["wtcs_pval"].to_numpy(dtype=float)), 5)
            df["spearman_fdr"] = np.round(benjamini_hochberg(df["spearman_pval"].to_numpy(dtype=float)), 5)
        return df

    rev_df = _score_set_collection(module_sets)
    rev_df.to_csv("results/results_dose_stratified_reversal.csv", index=False)

    indiv_rev_df = pd.DataFrame()
    if indiv_pathway_sets:
        indiv_rev_df = _score_set_collection(indiv_pathway_sets)
        indiv_rev_df.to_csv("results/results_individual_gobp_kegg_reactome_reversal.csv", index=False)

    # Dose-shift & 2-tier CysLT1R on-target proxy contrast table
    contrast_pairs = [
        ("DoseShift_AllCells", "Montelukast_Low", "Montelukast_High"),
        ("DoseShift_MatchedCells", "Montelukast_Low_MatchedCells", "Montelukast_High_MatchedCells"),
        ("ProxyContrast_vs_Zafirlukast_Low", "Zafirlukast_Low", "Montelukast_High"),
        ("ProxyContrast_vs_Zafirlukast_All", "Zafirlukast_All", "Montelukast_High"),
        ("ProxyContrast_vs_Pranlukast_Low", "Pranlukast_Low", "Montelukast_High"),
        ("ProxyContrast_vs_Pranlukast_All", "Pranlukast_All", "Montelukast_High"),
        ("ProxyContrast_vs_CYSLTR1_KD", "CYSLTR1_KD", "Montelukast_High"),
    ]
    delta_rows = []
    for mod_name in module_sets.keys():
        for c_lbl, ref_k, hi_k in contrast_pairs:
            r_ref = rev_df[(rev_df["module"] == mod_name) & (rev_df["perturbagen"] == ref_k)]
            r_hi = rev_df[(rev_df["module"] == mod_name) & (rev_df["perturbagen"] == hi_k)]
            if not r_ref.empty and not r_hi.empty:
                rr, rh = r_ref.iloc[0], r_hi.iloc[0]
                delta_rows.append({
                    "module": mod_name, "contrast": c_lbl, "ref_arm": ref_k, "high_arm": hi_k,
                    "WTCS_Ref": rr["wtcs"], "WTCS_High": rh["wtcs"],
                    "Delta_WTCS": round(float(rh["wtcs"] - rr["wtcs"]), 4) if pd.notna(rh["wtcs"]) and pd.notna(rr["wtcs"]) else np.nan,
                    "RawCS_Ref": rr["raw_cs"], "RawCS_High": rh["raw_cs"],
                    "Delta_RawCS": round(float(rh["raw_cs"] - rr["raw_cs"]), 4) if pd.notna(rh["raw_cs"]) and pd.notna(rr["raw_cs"]) else np.nan,
                    "Cos_Ref": rr["reversal_cosine"], "Cos_High": rh["reversal_cosine"],
                    "Delta_Cos": round(float(rh["reversal_cosine"] - rr["reversal_cosine"]), 4) if pd.notna(rh["reversal_cosine"]) and pd.notna(rr["reversal_cosine"]) else np.nan,
                })
    delta_df = pd.DataFrame(delta_rows)
    delta_df.to_csv("results/results_montelukast_dose_and_proxy_contrasts.csv", index=False)

    # -------------------------------------------------------------------------
    # STEP 6: Clinical Benchmark Alignment + L1000 MoA Mimicry + Gene Drivers
    # -------------------------------------------------------------------------
    print("\n[STEP 6/6] Computing clinical benchmark alignment, L1000 MoA mimicry & leading-edge gene drivers...")
    clin_rows = []
    core_mods = [m for m in ["Type2_Inflammation", "Epidermal_Barrier", "AMP_Innate"] if ad_submodules_norm.get(m)]
    universes = {
        "Full_Overlap": None,
        "AD_Consensus_Top500": cons_l1000.assign(abs_t=cons_l1000["mean_t_stat"].abs()).sort_values("abs_t", ascending=False).head(500).index.tolist(),
        "AD_All_Submodules": sorted(set().union(*(ad_submodules_norm[m] for m in core_mods))) if core_mods else [],
    }
    for m_name, m_genes in ad_submodules_norm.items():
        if m_genes:
            universes[m_name] = m_genes

    for bench_name, bench_df in clinical_reversal_degs.items():
        b_vec = bench_df.set_index("symbol")["log2FC"]
        for u_name, u_genes in universes.items():
            if u_genes is not None and len(u_genes) == 0:
                continue
            for sig_name in l1000_df.columns:
                cos_v, rho_v, sp_p, n_ov = safe_cosine_spearman(l1000_df[sig_name], b_vec, gene_subset=u_genes, min_n=5)
                clin_rows.append({
                    "clinical_benchmark": bench_name, "gene_universe": u_name, "perturbagen": sig_name,
                    "cosine_sim": cos_v, "spearman_rho": rho_v, "spearman_pval": sp_p, "n_genes": n_ov,
                })
    clin_sim_df = pd.DataFrame(clin_rows)
    if not clin_sim_df.empty:
        clin_sim_df["spearman_fdr"] = np.round(benjamini_hochberg(clin_sim_df["spearman_pval"].to_numpy(dtype=float)), 5)
    clin_sim_df.to_csv("results/results_clinical_benchmark_similarity.csv", index=False)

    # Pairwise L1000 MoA Mimicry (pan-cell + relevant-tissue strata)
    moa_agents = list(l1000_df.columns)
    moa_cos = pd.DataFrame(index=moa_agents, columns=moa_agents, dtype=float)
    moa_rho = pd.DataFrame(index=moa_agents, columns=moa_agents, dtype=float)
    for a in moa_agents:
        for b in moa_agents:
            if a == b:
                moa_cos.loc[a, b] = 1.0
                moa_rho.loc[a, b] = 1.0
            else:
                c_v, r_v, _, _ = safe_cosine_spearman(l1000_df[a], l1000_df[b], min_n=20)
                moa_cos.loc[a, b] = c_v
                moa_rho.loc[a, b] = r_v
    moa_cos.to_csv("results/results_l1000_moa_mimicry_cosine.csv")
    moa_rho.to_csv("results/results_l1000_moa_mimicry_spearman.csv")

    # Spotlight gene matrix
    spot_present = [g for g in SPOTLIGHT_GENES if g in l1000_df.index or g in consensus_filt.index]
    spot_cols = {"AD_Consensus_log2FC": consensus_filt["mean_log2FC"]}
    for b_lbl, b_df in clinical_reversal_degs.items():
        short_lbl = b_lbl.replace("_GSE130588", "").replace("_GSE133477", "").replace("_GSE157194", "")
        spot_cols[short_lbl] = b_df.set_index("symbol")["log2FC"]
    for s_lbl in [c for c in l1000_df.columns if "__" not in c]:
        spot_cols[f"L1000_{s_lbl}"] = l1000_df[s_lbl]
    spot_df = pd.DataFrame(spot_cols).reindex(spot_present).round(3)
    spot_df.index.name = "gene"
    spot_df.to_csv("results/results_spotlight_genes.csv")

    # -------------------------------------------------------------------------
    # FIGURES
    # -------------------------------------------------------------------------
    try:
        piv_wtcs = rev_df.pivot(index="perturbagen", columns="module", values="raw_cs")
        fig, ax = plt.subplots(figsize=(10.5, max(4.5, 0.42 * len(piv_wtcs))))
        im = ax.imshow(piv_wtcs.fillna(0).values, cmap="RdBu_r", vmin=-0.8, vmax=0.8, aspect="auto")
        ax.set_xticks(range(len(piv_wtcs.columns)))
        ax.set_xticklabels(piv_wtcs.columns, rotation=25, ha="right", fontsize=9)
        ax.set_yticks(range(len(piv_wtcs.index)))
        ax.set_yticklabels(piv_wtcs.index, fontsize=8)
        for i in range(len(piv_wtcs.index)):
            for j in range(len(piv_wtcs.columns)):
                val = piv_wtcs.iloc[i, j]
                if pd.notna(val):
                    ax.text(j, i, f"{val:+.2f}", ha="center", va="center", fontsize=7.5, color="black")
        plt.colorbar(im, ax=ax, label="Raw Connectivity Score (negative = AD reversal)")
        ax.set_title("Study 1 (v2): Dose- & Tissue-Stratified Reversal vs. Standardized AD Modules")
        plt.tight_layout()
        fig.savefig("results/fig1_dose_tissue_stratified_reversal.png", dpi=180)
        plt.close(fig)

        if not clin_sim_df.empty:
            sub_c = clin_sim_df[clin_sim_df["gene_universe"] == "AD_Consensus_Top500"]
            if not sub_c.empty:
                piv_c = sub_c.pivot(index="perturbagen", columns="clinical_benchmark", values="cosine_sim")
                fig, ax = plt.subplots(figsize=(10.5, max(4.5, 0.42 * len(piv_c))))
                im = ax.imshow(piv_c.fillna(0).values, cmap="coolwarm", vmin=-0.5, vmax=0.5, aspect="auto")
                ax.set_xticks(range(len(piv_c.columns)))
                ax.set_xticklabels(piv_c.columns, rotation=25, ha="right", fontsize=8)
                ax.set_yticks(range(len(piv_c.index)))
                ax.set_yticklabels(piv_c.index, fontsize=8)
                for i in range(len(piv_c.index)):
                    for j in range(len(piv_c.columns)):
                        val = piv_c.iloc[i, j]
                        if pd.notna(val):
                            ax.text(j, i, f"{val:+.2f}", ha="center", va="center", fontsize=7.5, color="black")
                plt.colorbar(im, ax=ax, label="Cosine Similarity to Clinical Reversal (Top 500 AD DEGs)")
                ax.set_title("Study 1 (v2): Alignment to Clinical Dupilumab & Crisaborole Reversal")
                plt.tight_layout()
                fig.savefig("results/fig2_clinical_benchmark_similarity.png", dpi=180)
                plt.close(fig)
    except Exception as e:
        qc_flags.append(f"FIGURE GENERATION WARNING: {e}")

    # -------------------------------------------------------------------------
    # VERY DETAILED MASTER SUMMARY REPORT (.txt) + ZIP EXPORT
    # -------------------------------------------------------------------------
    summary_path = Path("results/study1_montelukast_ad_master_summary.txt")

    def _hr(title="", char="=", width=104):
        return f"\n{char * width}\n{title}\n{char * width}" if title else char * width

    elapsed = time.time() - t_start
    lines = [
        "=" * 104,
        "STUDY 1 (v2 ENHANCED): DOSE- & TISSUE-STRATIFIED MONTELUKAST TRANSCRIPTOMIC REVERSAL — MASTER REPORT",
        "=" * 104,
        f"Generated (UTC)              : {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}",
        f"Total Pipeline Runtime       : {elapsed:.1f} seconds",
        f"Pathway Standards            : Live GO:BP (Enrichr + QuickGO descendants) + KEGG + Reactome",
        f"Low / High Dose Cutoffs      : <= {LOW_DOSE_MAX_UM} uM  |  >= {HIGH_DOSE_MIN_UM} uM",
        f"Max Signatures / Stratum     : {MAX_SIGS_PER_STRATUM}",
        f"Global Consensus GSEA Top-N  : {TOP_N_DEG_GSEA} Up / {TOP_N_DEG_GSEA} Down",
        f"GSEA Null Permutations       : {GSEA_PERMUTATIONS} (vectorized NumPy + Benjamini-Hochberg FDR)",
        f"Relevant Tissue Cell Panel   : {', '.join(sorted(RELEVANT_TISSUE_CELLS))}",
        f"Hormone/Tumor Control Panel  : {', '.join(sorted(HORMONE_TUMOR_CELLS))}",
    ]

    # Section 1: Standardized Pathways
    lines.append(_hr("1. STANDARDIZED GO:BP + KEGG + REACTOME PATHWAY INVENTORY (PLAN 3)"))
    lines.append(pathway_inv_df.to_string(index=False))

    # Section 2: L1000 Inventory + Montelukast Dose Histogram
    lines.append(_hr("2. LINCS L1000 PERTURBAGEN, DOSE & TISSUE-STRATIFIED INVENTORY (PLANS 1 & 4)"))
    lines.append(inv_df.to_string(index=False))
    if not montelukast_dose_df.empty:
        lines.append("\n--- Montelukast Exact Dose-Bin x Tissue-Stratum Breakdown ---")
        ct = pd.crosstab(montelukast_dose_df["dose_bin"], montelukast_dose_df["tissue_stratum"], margins=True)
        lines.append(ct.to_string())
        lines.append("\n--- Montelukast Distinct Concentrations Observed (uM) ---")
        dc = montelukast_dose_df["dose_uM"].value_counts(dropna=False).sort_index()
        lines.append(dc.to_string())
    if dose_fallback_notes:
        lines.append("\n--- Plan 1 Fallback Status ---")
        for n in dose_fallback_notes:
            lines.append(f"  * {n}")

    # Section 3: GEO Cohort Audit + GSE130588 Verification
    lines.append(_hr("3. GEO AD COHORT AUDIT & GSE130588 STRUCTURED METADATA VERIFICATION (PLAN 2)"))
    lines.append(geo_audit_df.to_string(index=False))
    if not gse130588_diag_preview.empty:
        lines.append("\n--- GSE130588 Parsed Structured Metadata Sample (first 5 rows) ---")
        lines.append(gse130588_diag_preview.head(5).to_string())
        lines.append("\n--- GSE130588 Parsed Value Counts ---")
        for col in gse130588_diag_preview.columns[1:4]:
            vc = gse130588_diag_preview[col].value_counts().head(10).to_dict()
            lines.append(f"  {col}: {vc}")
    if scorad_corr_summary:
        lines.append("\n--- GSE130588 SCORAD vs. Standardized AD Sub-Module Correlation ---")
        lines.append(pd.DataFrame(scorad_corr_summary).to_string(index=False))

    # Section 4: Consensus AD Signature & Module Coverage
    lines.append(_hr("4. CONSENSUS AD SIGNATURE & STANDARDIZED SUB-MODULE GENE COVERAGE"))
    lines.append(f"Cohorts combined        : {', '.join(sorted(ad_disease_degs.keys()))} (min_cohorts={min_cohorts_req})")
    lines.append(f"Total consensus genes   : {len(consensus_filt):,} ({len(cons_l1000):,} measured in L1000)")
    for mod_name, m_dict in module_sets.items():
        n_up_l = len(set(m_dict["up"]) & l1000_genes)
        n_dn_l = len(set(m_dict["down"]) & l1000_genes)
        lines.append(f"  - {mod_name:24s}: {len(m_dict['all']):4d} total | {len(m_dict['up']):3d} AD-Up ({n_up_l:3d} in L1000) | {len(m_dict['down']):3d} AD-Down ({n_dn_l:3d} in L1000)")

    # Section 5: Primary Reversal Scores (with 2000-perm pval & BH-FDR)
    lines.append(_hr("5. DOSE- & TISSUE-STRATIFIED REVERSAL SCORES (MORE NEGATIVE = STRONGER AD REVERSAL)"))
    for mod_name in module_sets.keys():
        lines.append(f"\n--- Module: {mod_name} ---")
        sub = rev_df[rev_df["module"] == mod_name].sort_values("raw_cs", ascending=True)
        lines.append(sub.to_string(index=False))

    # Section 6: Individual Constituent Pathway Reversal
    if not indiv_rev_df.empty:
        lines.append(_hr("6. INDIVIDUAL GO:BP, KEGG & REACTOME PATHWAY REVERSAL (RAW CS & BH-FDR)"))
        piv_ind = indiv_rev_df.pivot(index="module", columns="perturbagen", values="raw_cs")
        cols_keep = [
            c for c in [
                "Montelukast_Low", "Montelukast_High", "Montelukast_High__RelevantTissue",
                "Zafirlukast_Low", "Pranlukast_Low", "CYSLTR1_KD",
                "Zileuton", "Roflumilast", "Roflumilast__RelevantTissue",
                "Rolipram", "Rolipram__RelevantTissue", "Apremilast", "Crisaborole",
            ] if c in piv_ind.columns
        ]
        if cols_keep:
            lines.append(piv_ind[cols_keep].round(4).to_string())

    # Section 7: Dose-Shift Delta & 2-Tier CysLT1R On-Target Proxy Contrasts
    lines.append(_hr("7. MONTELUKAST DOSE-SHIFT DELTA & 2-TIER CYSLTR1 ON-TARGET PROXY CONTRASTS (PLAN 1)"))
    if not delta_df.empty:
        lines.append(delta_df.to_string(index=False))
    else:
        lines.append("[INFO] No low-dose or CysLT1R proxy arms available for contrast calculation.")

    # Section 8: Tissue-Relevance Stratification Audit (Rolipram Paradox Check)
    lines.append(_hr("8. TISSUE-RELEVANCE STRATIFICATION AUDIT: RELEVANT TISSUE vs. HORMONE TUMOR (PLAN 4)"))
    tissue_rows = []
    for base_drug in ["Montelukast_High", "Montelukast_Low", "Zileuton", "Roflumilast", "Rolipram", "Apremilast", "CYSLTR1_KD"]:
        pan = rev_df[rev_df["perturbagen"] == base_drug].set_index("module")
        rel = rev_df[rev_df["perturbagen"] == f"{base_drug}__RelevantTissue"].set_index("module")
        hor = rev_df[rev_df["perturbagen"] == f"{base_drug}__HormoneTumor"].set_index("module")
        if rel.empty and hor.empty:
            continue
        for mod_name in module_sets.keys():
            tissue_rows.append({
                "perturbagen": base_drug, "module": mod_name,
                "PanCell_RawCS": pan.loc[mod_name, "raw_cs"] if mod_name in pan.index else np.nan,
                "RelevantTissue_RawCS": rel.loc[mod_name, "raw_cs"] if mod_name in rel.index else np.nan,
                "RelevantTissue_FDR": rel.loc[mod_name, "wtcs_fdr"] if mod_name in rel.index else np.nan,
                "HormoneTumor_RawCS": hor.loc[mod_name, "raw_cs"] if mod_name in hor.index else np.nan,
                "HormoneTumor_FDR": hor.loc[mod_name, "wtcs_fdr"] if mod_name in hor.index else np.nan,
            })
    if tissue_rows:
        t_df = pd.DataFrame(tissue_rows)
        t_df.to_csv("results/results_tissue_stratification_audit.csv", index=False)
        lines.append(t_df.to_string(index=False))
    else:
        lines.append("[INFO] Insufficient tissue-stratified arms (>=2 sigs) to build tissue audit table.")

    # Section 9: Clinical Benchmark Alignment (Dupilumab vs Crisaborole)
    lines.append(_hr("9. CLINICAL BENCHMARK ALIGNMENT (DUPILUMAB GSE130588/GSE157194 vs. CRISABOROLE GSE133477)"))
    if not clin_sim_df.empty:
        for u_name in ["AD_Consensus_Top500", "AD_All_Submodules", "Full_Overlap", "Type2_Inflammation", "Epidermal_Barrier", "AMP_Innate", "cAMP_PDE4_5LOX_Panel"]:
            sub = clin_sim_df[clin_sim_df["gene_universe"] == u_name]
            if not sub.empty:
                lines.append(f"\n--- Gene Universe: {u_name} (sorted by cosine_sim descending) ---")
                lines.append(sub.sort_values(["clinical_benchmark", "cosine_sim"], ascending=[True, False]).to_string(index=False))
    else:
        lines.append("[INFO] No clinical reversal benchmarks available.")

    # Section 10: Direct L1000 MoA Mimicry
    lines.append(_hr("10. DIRECT L1000 MECHANISTIC MIMICRY (MONTELUKAST vs. 5-LOXi, PDE4i & CYSLTR1 PROXIES)"))
    lines.append("--- Pairwise Cosine Similarity (Full L1000 Overlap) ---")
    lines.append(moa_cos.to_string())
    lines.append("\n--- Pairwise Spearman Rho (Full L1000 Overlap) ---")
    lines.append(moa_rho.to_string())

    # Section 11: Gene-Level Reversal & Exacerbation Drivers for Montelukast_High per Sub-Module
    lines.append(_hr("11. TOP GENE-LEVEL REVERSAL & EXACERBATION DRIVERS (MONTELUKAST_HIGH vs. CONSENSUS AD)"))
    if "Montelukast_High" in l1000_df.columns:
        mk_hi = l1000_df["Montelukast_High"].dropna()
        for mod_name in ["Type2_Inflammation", "Epidermal_Barrier", "AMP_Innate", "cAMP_PDE4_5LOX_Panel"]:
            if mod_name not in module_sets:
                continue
            ov = sorted(set(module_sets[mod_name]["all"]) & set(mk_hi.index) & set(ad_fc_vec.index))
            if not ov:
                continue
            drv = pd.DataFrame({
                "AD_log2FC": ad_fc_vec.reindex(ov),
                "Montelukast_High_Z": mk_hi.reindex(ov),
            }).dropna()
            drv["product"] = drv["AD_log2FC"] * drv["Montelukast_High_Z"]
            lines.append(f"\n  [{mod_name}] Top 8 Reversed Genes (opposite sign; most negative product):")
            for g, r in drv.sort_values("product").head(8).iterrows():
                lines.append(f"    {g:12s} | AD_log2FC={r['AD_log2FC']:+.3f} | MK_High_Z={r['Montelukast_High_Z']:+.3f} | prod={r['product']:+.3f}")
            lines.append(f"  [{mod_name}] Top 6 Exacerbated/Concordant Genes (same sign; most positive product):")
            for g, r in drv.sort_values("product", ascending=False).head(6).iterrows():
                lines.append(f"    {g:12s} | AD_log2FC={r['AD_log2FC']:+.3f} | MK_High_Z={r['Montelukast_High_Z']:+.3f} | prod={r['product']:+.3f}")

    # Section 12: Spotlight Marker Readout
    lines.append(_hr("12. SPOTLIGHT MARKER READOUT TABLE (TYPE 2, BARRIER, AMP & cAMP/PDE4/5-LOX)"))
    lines.append(spot_df.to_string())

    # Section 13: Automated QC & Audit Flags
    lines.append(_hr("13. AUTOMATED QC & METHODOLOGICAL AUDIT FLAGS"))
    if qc_flags:
        for i, msg in enumerate(qc_flags, 1):
            lines.append(f"  {i}. {msg}")
    else:
        lines.append("  [OK] No methodological or QC flags triggered.")

    # Section 14: Artifact Manifest
    lines.append(_hr("14. OUTPUT ARTIFACT MANIFEST"))
    summary_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    for p in sorted(Path("results").glob("*")):
        if p.is_file():
            lines.append(f"  - {p.name:56s} ({p.stat().st_size:>10,} bytes)")
    final_text = "\n".join(lines) + "\n"
    summary_path.write_text(final_text, encoding="utf-8")

    zip_path = Path("study1_montelukast_ad_v2_results.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(Path("results").glob("*")):
            if p.is_file():
                zf.write(p, arcname=p.name)

    print(f"\n[OK] Master summary written -> {summary_path} ({len(final_text):,} chars, {len(lines)} lines)")
    print(f"[OK] Zip archive created    -> {zip_path} ({zip_path.stat().st_size:,} bytes)")
    print("\n" + final_text[:3200] + ("\n... [truncated preview; open downloaded .txt for full report] ..." if len(final_text) > 3200 else ""))

    if DRIVE_EXPORT_DIR.strip() and IN_COLAB:
        dst = Path(DRIVE_EXPORT_DIR.strip())
        dst.mkdir(parents=True, exist_ok=True)
        for p in Path("results").glob("*"):
            if p.is_file():
                (dst / p.name).write_bytes(p.read_bytes())
        print(f"[OK] Copied results to Drive: {dst}")

    if IN_COLAB:
        from google.colab import files  # type: ignore
        files.download(str(summary_path))
        files.download(str(zip_path))

    return {
        "pathway_inventory": pathway_inv_df,
        "inventory": inv_df,
        "geo_audit": geo_audit_df,
        "reversal": rev_df,
        "individual_pathway_reversal": indiv_rev_df,
        "dose_proxy_contrasts": delta_df,
        "clinical_similarity": clin_sim_df,
        "moa_cosine": moa_cos,
        "spotlight": spot_df,
        "summary_path": str(summary_path),
    }


if __name__ == "__main__":
    run_study1_pipeline()

STUDY 1 (v2 ENHANCED): DOSE- & TISSUE-STRATIFIED MONTELUKAST L1000 vs. AD & BENCHMARKS

[STEP 1/6] Resolving standardized GO:BP + KEGG + Reactome pathways...
  - Loading standardized GO:BP + KEGG + Reactome pathway libraries...
    [OK] Type2_Inflammation    :  264 unique standardized genes
    [OK] Epidermal_Barrier     :  562 unique standardized genes
    [OK] AMP_Innate            :  229 unique standardized genes
    [OK] cAMP_PDE4_5LOX_Panel  :  289 unique standardized genes

[STEP 2/6] Querying SigCom LINCS L1000 (union pert_name + pert_id + smiles + CysLT1R proxies)...
  - Montelukast  (Test_Compound): 28 union signatures found
      -> Montelukast_Mid                 : 12,327 genes | 1/1 sigs (1 cells)
      -> Montelukast_High                : 12,327 genes | 26/26 sigs (8 cells)
      -> Montelukast_All                 : 12,327 genes | 27/28 sigs (9 cells)
      -> Montelukast_High__RelevantTissue: 12,327 genes | 7/7 sigs (3 cells)
      -> Montelukast_High__HormoneTumor  : 12,

/usr/local/lib/python3.13/dist-packages/scipy/stats/_axis_nan_policy.py:611: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, axis=axis, **kwds)
/usr/local/lib/python3.13/dist-packages/scipy/stats/_axis_nan_policy.py:611: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, axis=axis, **kwds)
/usr/local/lib/python3.13/dist-packages/scipy/stats/_axis_nan_policy.py:611: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, axis=axis, **kwds)


  [OK] GSE130588: 21,753 genes | LS_W0=51, NL_W0=42, Dupi_W0=26, W4=24, W16=18
  [OK] GSE133477: 14,752 genes | LS_D1=79, NL_D1=40, Crisa_D15=39
  [OK] GSE121212: 31,362 genes | AD_LS=21, AD_NL=27
  [OK] GSE157194: 33,458 genes | AL_m0=57, AN_m0=54

[STEP 4/6] Computing cross-cohort consensus AD signature & standardized module splits...

[STEP 5/6] Running vectorized 2000-permutation WTCS/GSEA + BH-FDR...

[STEP 6/6] Computing clinical benchmark alignment, L1000 MoA mimicry & leading-edge gene drivers...

[OK] Master summary written -> results/study1_montelukast_ad_master_summary.txt (273,377 chars, 179 lines)
[OK] Zip archive created    -> study1_montelukast_ad_v2_results.zip (12,934,174 bytes)

STUDY 1 (v2 ENHANCED): DOSE- & TISSUE-STRATIFIED MONTELUKAST TRANSCRIPTOMIC REVERSAL — MASTER REPORT
Generated (UTC)              : 2026-11-26 10:16:42 UTC
Total Pipeline Runtime       : 549.9 seconds
Pathway Standards            : Live GO:BP (Enrichr + QuickGO descendants) + KEGG + Reactome
L

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# The End